# Sistema Inteligente de Detecção e Classificação de Placas de Trânsito
### AED · Projeto Integrador · 2ª Etapa — **Checkpoint 1 (N1)**

**Pontifícia Universidade Católica de Goiás** — Escola Politécnica e de Artes
Curso de Ciência de Dados e Inteligência Artificial
Disciplina **CDI1021 — Visão Computacional (2026/2)** · Prof. Welington Júlio Dias Rodrigues

**Equipe:** Caio Henrique · Fernanda Andrade · Alisson Leonardo · Vitor Manoel

---

## O que este notebook entrega

Este caderno implementa, de ponta a ponta, o pipeline de Processamento Digital de Imagens
exigido no Checkpoint 1: a partir de uma imagem de via urbana, produz uma **máscara binária
estável** das placas de sinalização vertical e dela extrai **medidas objetivas** — contagem,
área, centroide, caixa envolvente e classe geométrica.

| Entregável do Checkpoint 1 | Onde está neste notebook |
|---|---|
| Código organizado em funções | Seções 3 a 5 — uma função por etapa do pipeline |
| Estrutura do dataset documentada | Seção 2 — inventário automático (quantidade, dimensões, fontes) |
| Correção de iluminação | Seção 3.2 — CLAHE em LAB e *top-hat* |
| Limiarização com justificativa técnica | Seções 3.5 e 6 — comparação objetiva entre global, Otsu, Otsu restrito e adaptativa |
| Limpeza morfológica dimensionada pelo dataset | Seções 3.6 e 5 — kernel derivado da escala real do objeto |
| Contornos com filtro por área mínima | Seção 3.7 — área mínima derivada do percentil das anotações |
| Diagrama da arquitetura | Seção 9 — inclui o ponto de entrada da IA na 2ª Etapa |
| Evidências visuais antes/depois | Seção 7 — painéis de todas as etapas em várias imagens |
| Registro dos parâmetros adotados | Seção 10 — `parametros_adotados.json` e tabela em Markdown |

## Como executar

1. **Runtime → Executar tudo** (`Ctrl+F9`). Não há ajuste manual obrigatório.
2. Na Seção 1 o notebook pede a sua **chave da API do Roboflow** (gratuita, em
   `roboflow.com` → *Settings* → *API Keys*). Se preferir não usar a API, existem dois
   caminhos alternativos documentados na mesma célula.
3. Ao final, a Seção 10 compacta tudo o que foi gerado em `outputs/` para download e
   versionamento no repositório.

> **Nota de reprodutibilidade.** Nenhum parâmetro deste pipeline foi ajustado "até ficar
> bonito" em uma imagem. Os três parâmetros críticos — limiar, tamanho de kernel e área
> mínima — são **calculados a partir das anotações do próprio dataset** (Seção 5) e a
> escolha do método de limiarização é decidida por **métrica objetiva** sobre uma amostra
> de validação (Seção 6).

---
## 0. Ambiente

Instala apenas o que não vem pré-instalado no Google Colab. Rodando localmente, use o
`requirements.txt` do repositório (`pip install -r requirements.txt`).

In [ ]:
import sys, subprocess, importlib

EM_COLAB = "google.colab" in sys.modules


def garantir_pacote(modulo: str, pacote: str | None = None) -> bool:
    """Importa `modulo`; se ausente, instala `pacote` (padrao: mesmo nome) via pip."""
    try:
        importlib.import_module(modulo)
        return True
    except ImportError:
        alvo = pacote or modulo
        print(f"[setup] instalando {alvo} ...")
        codigo = subprocess.call([sys.executable, "-m", "pip", "install", "-q", alvo])
        if codigo != 0:
            print(f"[setup] FALHA ao instalar {alvo}")
            return False
        importlib.invalidate_caches()
        return True


for mod, pkg in [("cv2", "opencv-python"), ("numpy", "numpy"), ("matplotlib", "matplotlib"),
                 ("pandas", "pandas"), ("yaml", "pyyaml"), ("roboflow", "roboflow")]:
    garantir_pacote(mod, pkg)

print("\nAmbiente:", "Google Colab" if EM_COLAB else "local")

In [ ]:
import json, math, os, random, re, shutil, time, zipfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "figure.facecolor": "white",
    "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "font.size": 9,
})

# Raiz do projeto: /content no Colab, pasta do notebook em execucao local.
RAIZ = Path("/content/aed_visao") if EM_COLAB else Path.cwd()
if not EM_COLAB and RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

DIR_DADOS = RAIZ / "data"
DIR_SAIDA = RAIZ / "outputs"
DIR_FIGURAS = DIR_SAIDA / "figuras"
DIR_DOCS = RAIZ / "docs"
for d in (DIR_DADOS, DIR_SAIDA, DIR_FIGURAS, DIR_DOCS):
    d.mkdir(parents=True, exist_ok=True)

print(f"OpenCV      {cv2.__version__}")
print(f"NumPy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"Python      {sys.version.split()[0]}")
print(f"\nRaiz do projeto : {RAIZ}")
print(f"Dados           : {DIR_DADOS}")
print(f"Saidas          : {DIR_SAIDA}")

---
## 1. Aquisição do dataset

**Fonte primária:** [Placas de Trânsito — `paic/placas-de-transito-1jw8i`, versão 2](https://universe.roboflow.com/paic/placas-de-transito-1jw8i/dataset/2)
(Roboflow Universe). Trata-se de uma base de sinalização vertical **brasileira**, anotada em
formato de caixa delimitadora com os códigos do CONTRAN (A-1a, R-1, R-19 etc.), o que
sustenta a justificativa normativa das faixas de cor adotadas na Seção 3.4.

O download exige uma chave gratuita da API do Roboflow. Três caminhos, em ordem de
preferência:

| Caminho | Quando usar |
|---|---|
| **A — SDK do Roboflow** | Padrão. Peça a chave em `roboflow.com` → *Settings* → *API Keys*. |
| **B — link direto de download** | Se a chave falhar. No botão *Download Dataset* da página do Roboflow, escolha `YOLOv8` → *show download code* → copie a URL `https://universe.roboflow.com/ds/...?key=...` e cole em `URL_DOWNLOAD_DIRETO`. |
| **C — pasta local** | Se o dataset já estiver baixado (ou para usar as imagens de captura própria da equipe). Aponte `PASTA_LOCAL` para a pasta que contém `data.yaml` ou as imagens. |

A célula seguinte concentra toda a configuração editável do notebook.

In [ ]:
# ------------------------------------------------------------------ configuracao
WORKSPACE = "paic"
PROJETO = "placas-de-transito-1jw8i"
VERSAO = 2
FORMATO = "yolov12"         # exporta images/ + labels/ (YOLO) + data.yaml

URL_DOWNLOAD_DIRETO = ""    # caminho B: cole aqui a URL "https://universe.roboflow.com/ds/...?key=..."
PASTA_LOCAL = ""            # caminho C: cole aqui o caminho de uma pasta ja baixada

# Amostragem usada nas secoes de calibracao e avaliacao (mantem o notebook rapido).
N_AMOSTRA_CALIBRACAO = 400  # imagens lidas para estimar escala do objeto
N_AMOSTRA_VALIDACAO = 250   # imagens por amostra (ajuste e validacao, disjuntas)
N_EVIDENCIAS_VISUAIS = 6    # imagens com painel antes/depois (o enunciado exige >= 3)

In [ ]:
def baixar_via_sdk(destino: Path) -> Path | None:
    """Caminho A: baixa a versao publica do Roboflow Universe usando o SDK oficial."""
    try:
        from getpass import getpass
        from roboflow import Roboflow
    except ImportError:
        print("[dados] pacote roboflow indisponivel")
        return None
    chave = os.environ.get("ROBOFLOW_API_KEY") or getpass("Chave da API do Roboflow: ").strip()
    if not chave:
        print("[dados] nenhuma chave informada")
        return None
    try:
        rf = Roboflow(api_key=chave)
        projeto = rf.workspace(WORKSPACE).project(PROJETO)
        conjunto = projeto.version(VERSAO).download(FORMATO, location=str(destino))
        return Path(conjunto.location)
    except Exception as erro:                                  # noqa: BLE001
        print(f"[dados] SDK falhou: {type(erro).__name__}: {erro}")
        return None


def baixar_via_url(url: str, destino: Path) -> Path | None:
    """Caminho B: baixa e extrai o zip gerado pelo botao *Download Dataset*."""
    if not url:
        return None
    destino.mkdir(parents=True, exist_ok=True)
    zip_local = destino / "roboflow.zip"
    print("[dados] baixando zip ...")
    codigo = subprocess.call(["curl", "-sSL", "-o", str(zip_local), url])
    if codigo != 0 or not zip_local.exists():
        print("[dados] download direto falhou")
        return None
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(destino)
    zip_local.unlink(missing_ok=True)
    return destino


def localizar_dataset(raiz: Path) -> Path | None:
    """Procura recursivamente a pasta que contem `data.yaml` ou, na falta dele, imagens."""
    raiz = Path(raiz)
    if not raiz.exists():
        return None
    candidatos = sorted(raiz.rglob("data.yaml"))
    if candidatos:
        return candidatos[0].parent
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        achado = next(iter(raiz.rglob(ext)), None)
        if achado is not None:
            return raiz
    return None


def obter_dataset() -> Path:
    """Resolve a origem do dataset seguindo a ordem C -> ja baixado -> A -> B."""
    if PASTA_LOCAL:
        achado = localizar_dataset(Path(PASTA_LOCAL))
        if achado:
            print(f"[dados] usando pasta local: {achado}")
            return achado
        raise FileNotFoundError(f"PASTA_LOCAL nao contem imagens: {PASTA_LOCAL}")

    achado = localizar_dataset(DIR_DADOS)
    if achado:
        print(f"[dados] dataset ja presente em: {achado}")
        return achado

    alvo = DIR_DADOS / f"{PROJETO}-{VERSAO}"
    for tentativa in (lambda: baixar_via_sdk(alvo),
                      lambda: baixar_via_url(URL_DOWNLOAD_DIRETO, alvo)):
        resultado = tentativa()
        if resultado:
            achado = localizar_dataset(resultado) or localizar_dataset(DIR_DADOS)
            if achado:
                print(f"[dados] dataset disponivel em: {achado}")
                return achado

    raise FileNotFoundError(
        "Nao foi possivel obter o dataset.\n"
        "  - Caminho A: informe uma chave valida da API do Roboflow;\n"
        "  - Caminho B: preencha URL_DOWNLOAD_DIRETO na celula de configuracao;\n"
        "  - Caminho C: preencha PASTA_LOCAL com uma pasta que contenha as imagens."
    )

In [ ]:
DIR_DATASET = obter_dataset()

ARQ_YAML = DIR_DATASET / "data.yaml"
if ARQ_YAML.exists():
    META = yaml.safe_load(ARQ_YAML.read_text(encoding="utf-8"))
    CLASSES = list(META.get("names", []) or [])
    print(f"data.yaml encontrado | {len(CLASSES)} classes anotadas")
    print("Primeiras classes:", CLASSES[:8])
else:
    META, CLASSES = {}, []
    print("Aviso: data.yaml ausente. O inventario segue, mas sem os nomes das classes.")

---
## 2. Inventário e caracterização do dataset

O Checkpoint 1 exige a **estrutura de pastas do dataset documentada: quantidade de imagens,
dimensões e fontes**. As funções abaixo produzem esse inventário automaticamente e o
persistem em `outputs/inventario_dataset.csv`, de modo que o número citado no README e no
relatório seja sempre o número real da base versionada — não um valor digitado à mão.

O export YOLO do Roboflow organiza os dados como:

```
<dataset>/
├── data.yaml            # nomes das classes e caminhos dos splits
├── train/  images/  labels/
├── valid/  images/  labels/
└── test/   images/  labels/
```

Cada arquivo `labels/<nome>.txt` traz uma linha por objeto anotado, no formato YOLO
normalizado: `classe  cx  cy  largura  altura`, com todos os valores em `[0, 1]`.

In [ ]:
EXTENSOES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def listar_imagens(raiz: Path) -> list[Path]:
    """Todos os arquivos de imagem sob `raiz`, em ordem estavel."""
    return sorted(p for p in Path(raiz).rglob("*") if p.suffix.lower() in EXTENSOES)


def caminho_rotulo(img: Path) -> Path:
    """Caminho do .txt YOLO correspondente a uma imagem (troca images/ por labels/)."""
    partes = list(img.parts)
    for i in range(len(partes) - 1, -1, -1):
        if partes[i] == "images":
            partes[i] = "labels"
            break
    return Path(*partes).with_suffix(".txt")


def ler_rotulos(img: Path) -> list[tuple[int, float, float, float, float]]:
    """Le as anotacoes YOLO normalizadas de uma imagem. Lista vazia se nao houver."""
    arq = caminho_rotulo(img)
    if not arq.exists():
        return []
    itens = []
    for linha in arq.read_text(encoding="utf-8", errors="ignore").splitlines():
        campos = linha.split()
        if len(campos) < 5:
            continue
        try:
            itens.append((int(float(campos[0])), *(float(v) for v in campos[1:5])))
        except ValueError:
            continue
    return itens


def dimensoes(img: Path) -> tuple[int, int]:
    """(largura, altura) reais da imagem. (0, 0) se o arquivo nao decodificar."""
    dados = np.fromfile(str(img), dtype=np.uint8)
    mat = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if mat is None:
        return (0, 0)
    return (mat.shape[1], mat.shape[0])


def inventariar(raiz: Path, amostra_dimensoes: int = 400) -> pd.DataFrame:
    """Monta o inventario: split, arquivo, dimensoes e numero de objetos anotados."""
    imagens = listar_imagens(raiz)
    if not imagens:
        raise FileNotFoundError(f"Nenhuma imagem encontrada em {raiz}")
    passo = max(1, len(imagens) // amostra_dimensoes)
    linhas = []
    for i, img in enumerate(imagens):
        rotulos = ler_rotulos(img)
        partes = set(img.parts)
        split = next((s for s in ("train", "valid", "val", "test") if s in partes), "unico")
        larg, alt = dimensoes(img) if i % passo == 0 else (np.nan, np.nan)
        linhas.append({
            "arquivo": img.name, "caminho": str(img), "split": split,
            "largura": larg, "altura": alt,
            "n_objetos": len(rotulos),
            "classes": ",".join(sorted({str(r[0]) for r in rotulos})),
        })
    return pd.DataFrame(linhas)

In [ ]:
INVENTARIO = inventariar(DIR_DATASET)
INVENTARIO.to_csv(DIR_SAIDA / "inventario_dataset.csv", index=False, encoding="utf-8")

dim = INVENTARIO.dropna(subset=["largura"])
resumo_split = (INVENTARIO.groupby("split")
                .agg(imagens=("arquivo", "size"),
                     objetos=("n_objetos", "sum"),
                     objetos_por_imagem=("n_objetos", "mean"))
                .round(2))

print("=" * 66)
print("INVENTARIO DO DATASET")
print("=" * 66)
print(f"Fonte            : Roboflow Universe / {WORKSPACE}/{PROJETO} v{VERSAO}")
print(f"Local            : {DIR_DATASET}")
print(f"Imagens          : {len(INVENTARIO)}")
print(f"Objetos anotados : {int(INVENTARIO['n_objetos'].sum())}")
print(f"Classes          : {len(CLASSES)}")
if len(dim):
    print(f"Dimensoes        : {int(dim['largura'].median())} x {int(dim['altura'].median())} px "
          f"(mediana de {len(dim)} imagens medidas)")
    print(f"  minimo         : {int(dim['largura'].min())} x {int(dim['altura'].min())} px")
    print(f"  maximo         : {int(dim['largura'].max())} x {int(dim['altura'].max())} px")
print(f"Sem anotacao     : {int((INVENTARIO['n_objetos'] == 0).sum())} imagens")
print("=" * 66)
resumo_split

### 2.1. Escala do objeto na cena

Esta é a medida que **dimensiona os parâmetros morfológicos**. Em vez de escolher um kernel
`5×5` por hábito, medimos qual é o tamanho típico de uma placa no dataset e derivamos dele o
elemento estruturante e a área mínima de contorno (Seção 5).

A fração de área ocupada também diagnostica a natureza da base: se a mediana for muito alta,
as imagens são recortes centralizados na placa e não cenas completas — o que mudaria a
leitura dos resultados de segmentação.

In [ ]:
def estatisticas_de_escala(inventario: pd.DataFrame, n: int, largura_trabalho: int) -> pd.DataFrame:
    """Tamanho das caixas anotadas reescalado para a largura de trabalho do pipeline."""
    com_objeto = inventario[inventario["n_objetos"] > 0]
    if com_objeto.empty:
        return pd.DataFrame()
    amostra = com_objeto.sample(min(n, len(com_objeto)), random_state=SEMENTE)
    linhas = []
    for caminho in amostra["caminho"]:
        img = Path(caminho)
        rotulos = ler_rotulos(img)
        if not rotulos:
            continue
        larg, alt = dimensoes(img)
        if not larg or not alt:
            continue
        escala = largura_trabalho / larg
        alt_trab = alt * escala
        for _, _, _, w, h in rotulos:
            w_px, h_px = w * largura_trabalho, h * alt_trab
            linhas.append({
                "largura_px": w_px, "altura_px": h_px,
                "area_px": w_px * h_px,
                "lado_equivalente_px": math.sqrt(max(w_px * h_px, 1e-9)),
                "fracao_area": w * h,
            })
    return pd.DataFrame(linhas)


LARGURA_TRABALHO = 640
ESCALA = estatisticas_de_escala(INVENTARIO, N_AMOSTRA_CALIBRACAO, LARGURA_TRABALHO)

if ESCALA.empty:
    print("Sem anotacoes: os parametros usarao os valores padrao da Secao 4.")
else:
    q = ESCALA[["lado_equivalente_px", "area_px", "fracao_area"]].describe(
        percentiles=[0.05, 0.10, 0.25, 0.5, 0.75, 0.95]).round(2)
    print(f"{len(ESCALA)} caixas anotadas medidas, reescaladas para largura {LARGURA_TRABALHO} px\n")
    print(q.to_string())
    mediana_fracao = ESCALA["fracao_area"].median()
    print(f"\nA placa mediana ocupa {mediana_fracao * 100:.2f}% da area da imagem.")
    if mediana_fracao > 0.35:
        print("DIAGNOSTICO: base dominada por recortes proximos da placa, nao por cenas completas.")
    else:
        print("DIAGNOSTICO: cenas completas — ha, de fato, segmentacao a ser realizada.")

In [ ]:
if not ESCALA.empty:
    fig, eixos = plt.subplots(1, 3, figsize=(13, 3.4))
    eixos[0].hist(ESCALA["lado_equivalente_px"], bins=40, color="#2d6a9f", edgecolor="white")
    eixos[0].axvline(ESCALA["lado_equivalente_px"].median(), color="#c1440e", lw=2,
                     label=f"mediana {ESCALA['lado_equivalente_px'].median():.0f} px")
    eixos[0].set_title("Lado equivalente da placa"); eixos[0].set_xlabel("px"); eixos[0].legend()

    eixos[1].hist(ESCALA["area_px"], bins=40, color="#2d6a9f", edgecolor="white")
    p5 = ESCALA["area_px"].quantile(0.05)
    eixos[1].axvline(p5, color="#c1440e", lw=2, label=f"percentil 5 = {p5:.0f} px²")
    eixos[1].set_title("Área da caixa anotada"); eixos[1].set_xlabel("px²")
    eixos[1].set_yscale("log"); eixos[1].legend()

    eixos[2].hist(ESCALA["fracao_area"] * 100, bins=40, color="#2d6a9f", edgecolor="white")
    eixos[2].set_title("Fração da imagem ocupada"); eixos[2].set_xlabel("%")
    eixos[2].set_yscale("log")

    fig.suptitle("Escala do objeto no dataset — base de dimensionamento dos parâmetros", y=1.04)
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "00_escala_do_objeto.png", bbox_inches="tight")
    plt.show()

---
## 3. O pipeline, etapa por etapa

Cada etapa do pipeline é uma função pura e independente, testável isoladamente. O
orquestrador da Seção 4 apenas as encadeia. A ordem não é arbitrária:

```
imagem → redimensionar → corrigir iluminação → suavizar → mapa de evidência cromática
       → limiarizar → morfologia → contornos → descritores
```

Três decisões de ordem merecem registro:

1. **Suavizar antes de limiarizar.** O ruído de alta frequência polui o histograma e desloca
   o limiar de Otsu. Segmentar sem suavizar e concluir que "Otsu não funciona" é justamente
   o erro que a orientação da AED aponta. A Seção 6 mede o tamanho desse efeito neste
   dataset — e o resultado tem uma nuance que vale ler.
2. **Corrigir iluminação antes de converter para HSV.** Contraluz e sombra deprimem o canal
   `V`; o CLAHE reequilibra a luminância *sem* alterar a matiz, preservando a assinatura de
   cor normativa da placa.
3. **Contornos a partir da máscara morfológica, nunca do Canny.** O Canny devolve bordas de
   um pixel de espessura; alimentar `findContours` com elas produz um contorno interno e
   outro externo para o mesmo objeto e **duplica a contagem**. O Canny aparece na Seção 3.8
   apenas como evidência visual.

### 3.1. Parâmetros e faixas normativas de cor

`Parametros` reúne, em um único objeto serializável, tudo o que governa o pipeline. Os
valores aqui são apenas o ponto de partida: a Seção 5 sobrescreve `k_abertura`,
`k_fechamento` e `area_minima` com valores medidos no dataset, e a Seção 6 escolhe
`metodo_limiar` por métrica.

As faixas de matiz correspondem às cores normativas da sinalização vertical brasileira
(Resolução CONTRAN nº 160/2004 e Manual Brasileiro de Sinalização de Trânsito):

| Cor | Uso normativo | Centro de matiz (H do OpenCV, 0–179) |
|---|---|---|
| Vermelho | Regulamentação (R-1 "Pare", R-19 "Velocidade máxima", orlas) | 0 — a matiz vermelha é circular e envolve 0 e 179 |
| Amarelo | Advertência (placas A-*, losangulares) | 27 |
| Azul | Indicação e serviços auxiliares | 112 |

**O verde é deliberadamente excluído.** Vegetação urbana ocupa exatamente a mesma faixa de
matiz das placas de indicação verdes e, em cenas de via, cobre uma área ordens de grandeza
maior. Incluí-lo inundaria a máscara de falsos positivos. Essa restrição é uma limitação
consciente do escopo do Checkpoint 1, registrada na Seção 11.

In [ ]:
# Faixas de matiz das cores normativas. `h_centro` e `h_sigma` estao na escala H do
# OpenCV (0..179); `s_min`/`v_min` sao as portas minimas de saturacao e valor que
# descartam pixels acinzentados e pixels escuros demais para terem matiz confiavel.
FAIXAS_CONTRAN = [
    {"nome": "vermelho", "h_centro": 0.0,   "h_sigma": 9.0,  "s_min": 80, "v_min": 50},
    {"nome": "amarelo",  "h_centro": 27.0,  "h_sigma": 8.0,  "s_min": 90, "v_min": 80},
    {"nome": "azul",     "h_centro": 112.0, "h_sigma": 12.0, "s_min": 80, "v_min": 45},
]


@dataclass
class Parametros:
    """Configuracao completa de uma execucao do pipeline."""
    # entrada
    largura_trabalho: int = 640
    # etapa A - iluminacao
    usar_clahe: bool = True
    clahe_clip: float = 2.0
    clahe_grade: int = 8
    # etapa B - suavizacao
    suavizacao: str = "gaussiano"        # "gaussiano" | "mediana" | "bilateral" | "nenhuma"
    suavizacao_k: int = 5
    # etapa C - evidencia cromatica
    faixas: list = field(default_factory=lambda: [dict(f) for f in FAIXAS_CONTRAN])
    # etapa D - limiarizacao
    metodo_limiar: str = "otsu_restrito"  # "global" | "otsu" | "otsu_restrito" | "adaptativa"
    limiar_global: int = 96
    adapt_bloco: int = 51
    adapt_c: int = -10
    # etapa E - morfologia
    k_abertura: int = 3
    k_fechamento: int = 11
    preencher_buracos: bool = True
    # etapa F - contornos e filtros de forma
    area_minima: int = 200
    area_maxima: int = 0                  # 0 = sem limite superior
    razao_aspecto: tuple = (0.35, 2.85)
    extensao_minima: float = 0.35
    solidez_minima: float = 0.70

    def para_dict(self) -> dict:
        d = asdict(self)
        d["razao_aspecto"] = list(self.razao_aspecto)
        return d


def impar(n, minimo: int = 3) -> int:
    """Arredonda para o inteiro impar >= `minimo` (kernels do OpenCV exigem lado impar)."""
    n = int(round(n))
    n = max(n, minimo)
    return n if n % 2 == 1 else n + 1

### 3.2. Entrada e redimensionamento

Padronizar a largura em 640 px cumpre dois papéis: torna o custo por imagem previsível e,
sobretudo, torna os parâmetros em pixels **comparáveis entre imagens de resoluções
diferentes**. Um kernel de 11 px significa algo completamente distinto em uma imagem de
640 px e em uma de 1920 px; fixando a largura, um único conjunto de parâmetros vale para
toda a base.

`carregar_imagem` usa `np.fromfile` + `cv2.imdecode` em vez de `cv2.imread` porque este
último falha silenciosamente com caminhos que contenham acentos ou espaços — situação comum
no Windows e no Google Drive montado.

In [ ]:
def carregar_imagem(caminho) -> np.ndarray:
    """Le uma imagem do disco e devolve um array RGB uint8."""
    dados = np.fromfile(str(caminho), dtype=np.uint8)
    bgr = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Nao foi possivel decodificar a imagem: {caminho}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def redimensionar(rgb: np.ndarray, largura: int = 640) -> tuple[np.ndarray, float]:
    """Reescala preservando a proporcao. Devolve (imagem, fator de escala aplicado)."""
    altura_orig, largura_orig = rgb.shape[:2]
    if largura_orig == largura:
        return rgb.copy(), 1.0
    escala = largura / float(largura_orig)
    novo = (largura, max(1, int(round(altura_orig * escala))))
    # INTER_AREA e o reamostrador correto para reducao: media os pixels da vizinhanca
    # em vez de amostra-los, o que evita aliasing nas bordas da placa.
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_LINEAR
    return cv2.resize(rgb, novo, interpolation=interp), escala

### 3.3. Etapa A — Correção de iluminação

Duas técnicas, com papéis distintos:

**CLAHE (`Contrast Limited Adaptive Histogram Equalization`)** — aplicado **apenas ao canal
`L` do espaço LAB**, e não aos três canais RGB. A razão é decisiva para este projeto:
equalizar RGB canal a canal altera as proporções entre eles e, portanto, **desloca a matiz**
— exatamente o atributo em que a segmentação se apoia. No LAB, `L` carrega a luminância e
`a`/`b` a cromaticidade; corrigir `L` isoladamente reequilibra a iluminação preservando a
cor normativa da placa.

O `clipLimit = 2.0` limita a amplificação de contraste por bloco. Valores altos (≥ 4)
saturam o céu e realçam ruído em regiões homogêneas; a grade `8×8` dá blocos de 80×60 px em
uma imagem de 640 px de largura, ou seja, cerca de duas vezes o tamanho de uma placa típica
— grande o bastante para capturar a variação de iluminação local sem tratar a própria placa
como "fundo".

**Top-hat** — a diferença entre a imagem e sua abertura morfológica isola estruturas claras
menores que o elemento estruturante, removendo o gradiente lento de fundo. É a ferramenta
adequada quando o problema é *sombra projetada extensa*. Fica disponível como alternativa
e é exibida na comparação visual, mas não integra o pipeline padrão: sobre imagem colorida
ele obriga a trabalhar em tons de cinza, o que descartaria a informação de cor.

In [ ]:
def corrigir_iluminacao_clahe(rgb: np.ndarray, clip: float = 2.0, grade: int = 8) -> np.ndarray:
    """CLAHE sobre o canal L do LAB — corrige luminancia sem deslocar a matiz."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    canal_l, canal_a, canal_b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grade), int(grade)))
    lab_corrigido = cv2.merge([clahe.apply(canal_l), canal_a, canal_b])
    return cv2.cvtColor(lab_corrigido, cv2.COLOR_LAB2RGB)


def corrigir_iluminacao_tophat(rgb: np.ndarray, k: int = 25) -> np.ndarray:
    """Top-hat em tons de cinza: realca estruturas claras menores que o elemento."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k), impar(k)))
    return cv2.morphologyEx(cinza, cv2.MORPH_TOPHAT, elemento)

### 3.4. Etapa B — Suavização

O filtro **gaussiano 5×5** é o padrão do pipeline. Ele atenua o ruído de sensor e a textura
fina de asfalto e vegetação — as duas fontes que mais fragmentam a máscara — antes que essa
alta frequência contamine o histograma usado pela limiarização.

O tamanho 5 não é arbitrário: como as placas medianas do dataset têm dezenas de pixels de
lado (Seção 2.1), um kernel de 5 px remove estruturas muito menores que o objeto de
interesse sem borrar sua orla. Kernels maiores começam a comer a borda das placas pequenas
e distantes, que são justamente as mais difíceis de recuperar.

A **mediana** fica disponível para ruído impulsivo (*sal e pimenta*), no qual o gaussiano
apenas espalha o pixel corrompido em vez de removê-lo; e o **bilateral**, mais caro, para
quando se quiser suavizar preservando bordas.

> **Atenção ao que a medição diz.** A ablação da Seção 6 **não** confirma ganho da suavização
> *neste* dataset — desligá-la sai ligeiramente à frente. A explicação é a origem das imagens:
> o export do Roboflow já passou por redimensionamento e recompressão JPEG, etapas que atenuam
> boa parte do ruído de sensor que o filtro existiria para remover; o que sobra do gaussiano é
> a erosão das placas menores. O filtro é mantido no pipeline por duas razões declaradas —
> é etapa exigida pelo Checkpoint 1, e as imagens de captura autoral da equipe, ainda a
> incorporar, chegam sem esse pré-tratamento. A conclusão registrada é a da medição, não a da
> expectativa.

In [ ]:
def suavizar(rgb: np.ndarray, metodo: str = "gaussiano", k: int = 5) -> np.ndarray:
    """Reducao de ruido antes da limiarizacao."""
    if metodo == "nenhuma":
        return rgb.copy()
    k = impar(k)
    if metodo == "gaussiano":
        return cv2.GaussianBlur(rgb, (k, k), 0)
    if metodo == "mediana":
        return cv2.medianBlur(rgb, k)
    if metodo == "bilateral":
        return cv2.bilateralFilter(rgb, k, 75, 75)
    raise ValueError(f"metodo de suavizacao desconhecido: {metodo}")

### 3.5. Etapa C — Mapa de evidência cromática

Esta é a etapa que transforma um problema de cor em um problema de limiarização, e a decisão
de projeto mais importante do pipeline.

Limiarizar diretamente a imagem em tons de cinza não funciona: uma placa vermelha e o
asfalto ao lado podem ter a **mesma luminância**. O que distingue a placa é a combinação de
*matiz normativa* com *alta saturação*. Então, em vez de aplicar uma máscara binária por
faixa de HSV — que exige escolher bordas rígidas e produz um resultado frágil —, construímos
um **mapa escalar contínuo** em `[0, 255]`, no qual o valor de cada pixel mede quanto ele se
parece com a cor de uma placa:

$$E(x,y) \;=\; \underbrace{\max_{c \,\in\, \{\text{verm.},\,\text{amar.},\,\text{azul}\}} \exp\!\left(-\frac{d_H(H_{x,y},\,\mu_c)^2}{2\sigma_c^2}\right) \cdot \mathbb{1}\!\left[S_{x,y} \ge S_c^{\min} \wedge V_{x,y} \ge V_c^{\min}\right]}_{\text{proximidade da matiz normativa}} \;\times\; \underbrace{\frac{S_{x,y}}{255}}_{\text{pureza da cor}}$$

Três propriedades tornam esse mapa adequado:

- **`d_H` é a distância circular de matiz.** O vermelho vive nas duas pontas da escala `H`
  (perto de 0 e de 179); tratar `H` como um eixo linear partiria a placa vermelha em duas.
- **O peso gaussiano substitui bordas rígidas.** Um pixel desbotado, de matiz um pouco
  desviada, não é descartado — recebe peso menor e a decisão final fica com a limiarização,
  que enxerga a distribuição inteira da imagem.
- **A multiplicação por `S` penaliza o fundo acinzentado.** Concreto, asfalto e céu têm
  saturação baixa e caem para perto de zero mesmo quando sua matiz é acidentalmente próxima
  de uma cor normativa.

O resultado é uma imagem de um único canal, com o objeto de interesse concentrado na cauda
alta do histograma — o insumo natural para a etapa de limiarização.

In [ ]:
def distancia_circular_matiz(h: np.ndarray, centro: float, periodo: float = 180.0) -> np.ndarray:
    """Distancia entre matizes num eixo circular: |H - centro| pelo caminho mais curto.

    Necessaria porque o vermelho ocupa as duas extremidades da escala H do OpenCV:
    H = 178 esta a 2 unidades de H = 0, e nao a 178.
    """
    d = np.abs(h.astype(np.float32) - float(centro))
    return np.minimum(d, periodo - d)


def mapa_evidencia_cromatica(rgb: np.ndarray, faixas: list | None = None) -> np.ndarray:
    """Mapa escalar uint8: quanto cada pixel se parece com a cor de uma placa."""
    faixas = faixas if faixas is not None else FAIXAS_CONTRAN
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    canal_h, canal_s, canal_v = cv2.split(hsv)

    peso = np.zeros(canal_h.shape, dtype=np.float32)
    for faixa in faixas:
        d = distancia_circular_matiz(canal_h, faixa["h_centro"])
        proximidade = np.exp(-(d ** 2) / (2.0 * float(faixa["h_sigma"]) ** 2))
        porta = ((canal_s >= faixa["s_min"]) & (canal_v >= faixa["v_min"])).astype(np.float32)
        peso = np.maximum(peso, proximidade * porta)

    mapa = peso * (canal_s.astype(np.float32) / 255.0)
    return np.clip(mapa * 255.0, 0, 255).astype(np.uint8)

### 3.6. Etapa D — Limiarização

Quatro métodos implementados e comparados objetivamente na Seção 6:

| Método | Como decide o limiar | Comportamento esperado neste problema |
|---|---|---|
| **Global fixo** | Constante escolhida a priori | Reprodutível e barato, mas cego à imagem: falha sob contraluz forte. |
| **Otsu** | Maximiza a variância entre classes sobre o histograma completo | Pressupõe histograma **bimodal**. Aqui o fundo domina — a placa costuma ocupar menos de 1% dos pixels —, então a moda do objeto quase não existe e o limiar tende a cair baixo demais. |
| **Otsu restrito** | Mesmo critério, mas sobre o histograma **apenas dos pixels com evidência > 0** | Descarta a massa de zeros do fundo antes de procurar o limiar, restaurando a bimodalidade que o método pressupõe. |
| **Adaptativa** | Limiar por vizinhança gaussiana local | Robusta a gradiente de iluminação, mas sobre um mapa esparso ela realça ruído local dentro de regiões de fundo uniformes. |

O **Otsu restrito** é a variante que trata a patologia real deste mapa. Ele é o padrão
proposto, mas a decisão final é da métrica: a Seção 6 mede `precisão`, `recall` e `F1` dos
quatro métodos contra as anotações do dataset e adota o vencedor.

`_otsu_1d` reimplementa o critério de Otsu em NumPy porque `cv2.threshold` só o aplica ao
histograma da imagem inteira — não há como restringi-lo a um subconjunto de pixels.

In [ ]:
def _otsu_1d(valores: np.ndarray) -> int:
    """Limiar de Otsu sobre um vetor de intensidades uint8 arbitrario.

    Maximiza a variancia entre classes sigma_b^2(t) = [mu_T*w(t) - mu(t)]^2 / [w(t)(1-w(t))],
    forma equivalente e numericamente estavel do criterio original.
    """
    hist = np.bincount(valores.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    if total == 0:
        return 127
    prob = hist / total
    niveis = np.arange(256)
    peso = np.cumsum(prob)                      # w(t)
    media = np.cumsum(prob * niveis)            # mu(t)
    media_total = media[-1]                     # mu_T
    denominador = peso * (1.0 - peso)
    with np.errstate(divide="ignore", invalid="ignore"):
        variancia_entre = (media_total * peso - media) ** 2 / denominador
    variancia_entre[~np.isfinite(variancia_entre)] = -1.0
    return int(np.argmax(variancia_entre))


def limiarizar(mapa: np.ndarray, metodo: str = "otsu_restrito", limiar_global: int = 96,
               adapt_bloco: int = 51, adapt_c: int = -10) -> tuple[np.ndarray, float]:
    """Binariza o mapa de evidencia. Devolve (mascara, limiar aplicado)."""
    if metodo == "global":
        limiar, mascara = cv2.threshold(mapa, int(limiar_global), 255, cv2.THRESH_BINARY)
        return mascara, float(limiar)

    if metodo == "otsu":
        limiar, mascara = cv2.threshold(mapa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return mascara, float(limiar)

    if metodo == "otsu_restrito":
        # Restringe o histograma aos pixels com alguma evidencia cromatica: sem isso,
        # a moda gigante de zeros do fundo domina a variancia e derruba o limiar.
        candidatos = mapa[mapa > 0]
        if candidatos.size < 50:
            return np.zeros_like(mapa), 255.0
        limiar = float(_otsu_1d(candidatos))
        return ((mapa > limiar) * 255).astype(np.uint8), limiar

    if metodo == "adaptativa":
        mascara = cv2.adaptiveThreshold(
            mapa, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
            impar(adapt_bloco), int(adapt_c))
        return mascara, float("nan")           # limiar e local, nao ha valor unico

    raise ValueError(f"metodo de limiarizacao desconhecido: {metodo}")

### 3.7. Etapa E — Limpeza morfológica

A máscara recém-binarizada tem dois defeitos previsíveis, e cada operação trata um deles:

**Abertura (erosão → dilatação), kernel pequeno.** Remove os fragmentos isolados de poucos
pixels — reflexos em lataria, luzes de freio, detalhes de fachada — que sobreviveram à
limiarização. O kernel precisa ser **menor que a menor placa que se pretende detectar**;
dimensioná-lo acima disso apaga o objeto junto com o ruído.

**Fechamento (dilatação → erosão), kernel maior.** Esta é a operação essencial do problema.
Uma placa de regulamentação é uma **orla vermelha em torno de um miolo branco**: o mapa de
evidência cromática enxerga o anel, não o disco. Sem fechamento, `findContours` devolveria um
anel fino, com área e centroide errados. O kernel precisa ser da ordem da **espessura do
miolo** para que as duas margens do anel se toquem e o objeto se torne sólido.

**Preenchimento de buracos.** Redesenha cada contorno externo preenchido, consolidando o que
o fechamento não fechou. Torna área, solidez e extensão descritores da placa inteira.

Os dois kernels são calculados a partir da escala real do objeto na Seção 5 — não escolhidos
por tentativa.

In [ ]:
def preencher_buracos(mascara: np.ndarray) -> np.ndarray:
    """Redesenha cada contorno externo preenchido, eliminando vazios internos."""
    contornos = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    cheia = np.zeros_like(mascara)
    if contornos:
        cv2.drawContours(cheia, contornos, -1, 255, thickness=cv2.FILLED)
    return cheia


def limpar_mascara(mascara: np.ndarray, k_abertura: int = 3, k_fechamento: int = 11,
                   preencher: bool = True) -> np.ndarray:
    """Abertura (remove ruido) -> fechamento (consolida a orla) -> preenchimento."""
    saida = mascara.copy()
    if k_abertura and k_abertura >= 3:
        elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k_abertura),) * 2)
        saida = cv2.morphologyEx(saida, cv2.MORPH_OPEN, elemento)
    if k_fechamento and k_fechamento >= 3:
        elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k_fechamento),) * 2)
        saida = cv2.morphologyEx(saida, cv2.MORPH_CLOSE, elemento)
    if preencher:
        saida = preencher_buracos(saida)
    return saida

### 3.8. Etapa F — Contornos, filtros e descritores geométricos

`cv2.findContours` recebe a **máscara morfológica**, com `RETR_EXTERNAL` — só os contornos
mais externos, o que já evita contar o miolo de uma placa como um segundo objeto.

#### Filtros aplicados a cada contorno

| Filtro | Rejeita |
|---|---|
| `área ≥ area_minima` | Ruído residual. Sem ele, o ruído entra na contagem — o segundo erro apontado na orientação da AED. |
| `área ≤ area_maxima` | **Céu, fachadas e maciços de vegetação.** Uma faixa de céu azul limpo tem a matiz normativa do azul, saturação alta e forma convexa — e passa por todos os outros filtros. Só a escala a separa de uma placa de indicação. Veja a Seção 5.2: o teto é real, mas o valor que maximiza F1 neste dataset ainda deixa passar parte desses casos. |
| `0,35 ≤ largura/altura ≤ 2,85` | Faixas, meios-fios e postes: estruturas muito alongadas. O intervalo cobre da placa mais larga (retangular de indicação) à mais alta. |
| `extensão ≥ 0,35` | Contornos rendilhados que ocupam pouco da própria caixa envolvente — vegetação, sombras recortadas. |
| `solidez ≥ 0,70` | Formas côncavas e fragmentadas. Toda placa normativa é convexa. |

Os dois limites de área são as **duas pontas do mesmo argumento**: a placa tem uma escala
característica na cena, medida na Seção 2.1, e o que estiver muito abaixo ou muito acima dela
não é placa. Ambos são derivados da distribuição de áreas anotadas na Seção 5.2.

#### Classificação geométrica

Os limiares abaixo **não foram estimados a olho**: cada forma normativa foi rasterizada em
cinco escalas (`r = 12` a `100 px`) e seus descritores medidos.

| Forma | Vértices (`approxPolyDP`, ε = 2,5% do perímetro) | Circularidade | Extensão | Área / círculo mínimo |
|---|---|---|---|---|
| Triangular (R-2 "Dê a preferência") | 3 | 0,55 | ~0,50 | — |
| Losangular (advertência A-*) | 4 | 0,76–0,78 | **~0,50** | — |
| Retangular (indicação) | 4 | 0,74 | **~1,00** | — |
| Circular (regulamentação) | ≥ 5 | 0,86–0,89 | ~0,79 | **0,93–0,99** |
| Octogonal (R-1 "Pare") | ≥ 5 | 0,95 | ~0,83 | **0,88–0,90** |

Duas escolhas decorrem diretamente dessa medição:

- **Losango vs. retângulo pela extensão, não pelo ângulo.** Um losango preenche metade da sua
  caixa envolvente alinhada aos eixos; um retângulo, praticamente toda ela. A alternativa
  natural seria o ângulo de `cv2.minAreaRect`, mas a convenção desse ângulo mudou entre
  OpenCV 4 e 5, o que quebraria o código conforme a versão do ambiente. A extensão é estável.
- **Círculo vs. octógono só é decidido em visada frontal.** A medição mostrou que, sob
  perspectiva oblíqua, nenhum descritor clássico separa as duas formas: a projeção achata
  ambas em elipses de razão indistinguível. Quando a elongação da elipse ajustada passa de
  1,15, o objeto é rotulado `circular` — classe majoritária, já que o R-1 é a única placa
  octogonal do CTB — e sinalizado com `forma_ambigua = True`, em vez de fingir uma certeza
  que a geometria não sustenta.

In [ ]:
def classificar_geometria(contorno, extensao: float, epsilon: float = 0.025) -> tuple[str, int, bool]:
    """Classe geometrica do contorno. Devolve (forma, n_vertices, forma_ambigua).

    Limiares calibrados sobre formas normativas rasterizadas em cinco escalas
    (r = 12..100 px) — ver tabela na celula anterior.
    """
    perimetro = cv2.arcLength(contorno, True)
    if perimetro <= 0:
        return "indefinido", 0, False

    vertices = len(cv2.approxPolyDP(contorno, epsilon * perimetro, True))

    if vertices == 3:
        return "triangular", vertices, False

    if vertices == 4:
        if extensao >= 0.72:
            return "retangular", vertices, False
        if extensao <= 0.62:
            return "losango", vertices, False
        return "quadrilatero", vertices, True

    if vertices >= 5:
        area = cv2.contourArea(contorno)
        _, raio = cv2.minEnclosingCircle(contorno)
        razao_circulo = area / (math.pi * raio ** 2) if raio > 0 else 0.0

        # Elongacao da elipse ajustada: mede o quanto a visada e obliqua.
        elongacao = 1.0
        if len(contorno) >= 5:
            (_, _), (eixo_maior, eixo_menor), _ = cv2.fitEllipse(contorno)
            if min(eixo_maior, eixo_menor) > 0:
                elongacao = max(eixo_maior, eixo_menor) / min(eixo_maior, eixo_menor)

        if elongacao > 1.15:
            # Sob perspectiva, circulo e octogono sao indistinguiveis por descritor
            # classico: assume-se a classe majoritaria e marca-se a ambiguidade.
            return "circular", vertices, True
        return ("circular" if razao_circulo >= 0.92 else "octogonal"), vertices, False

    return "indefinido", vertices, True


def descrever_contorno(contorno) -> dict:
    """Descritores geometricos de um contorno aceito."""
    area = float(cv2.contourArea(contorno))
    perimetro = float(cv2.arcLength(contorno, True))
    x, y, largura, altura = cv2.boundingRect(contorno)

    momentos = cv2.moments(contorno)
    cx = momentos["m10"] / momentos["m00"] if momentos["m00"] else x + largura / 2.0
    cy = momentos["m01"] / momentos["m00"] if momentos["m00"] else y + altura / 2.0

    circularidade = (4.0 * math.pi * area / perimetro ** 2) if perimetro > 0 else 0.0
    casco = cv2.convexHull(contorno)
    area_casco = float(cv2.contourArea(casco))
    solidez = area / area_casco if area_casco > 0 else 0.0
    extensao = area / float(largura * altura) if largura * altura else 0.0
    razao = largura / float(altura) if altura else 0.0

    forma, vertices, ambigua = classificar_geometria(contorno, extensao)
    if forma == "triangular":
        # Apice para baixo (R-2 "De a preferencia") desloca o centroide para cima
        # em relacao ao centro da caixa envolvente.
        forma = "triangular_invertido" if cy < y + altura / 2.0 else "triangular"

    return {
        "area_px": area, "perimetro_px": perimetro,
        "x": int(x), "y": int(y), "w": int(largura), "h": int(altura),
        "cx": float(cx), "cy": float(cy),
        "circularidade": float(circularidade), "solidez": float(solidez),
        "extensao": float(extensao), "razao_aspecto": float(razao),
        "vertices": int(vertices), "forma": forma, "forma_ambigua": bool(ambigua),
    }


def extrair_contornos(mascara: np.ndarray, area_minima: int = 200,
                      area_maxima: int = 0,
                      razao_aspecto: tuple = (0.35, 2.85),
                      extensao_minima: float = 0.35,
                      solidez_minima: float = 0.70) -> tuple[list, list, dict]:
    """Contornos externos filtrados por area e forma. Devolve (contornos, descritores, stats)."""
    brutos = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    aceitos, descritores = [], []
    motivos = {"area_min": 0, "area_max": 0, "aspecto": 0, "extensao": 0, "solidez": 0}

    for contorno in brutos:
        area = cv2.contourArea(contorno)
        if area < area_minima:
            motivos["area_min"] += 1
            continue
        if area_maxima and area > area_maxima:
            motivos["area_max"] += 1
            continue
        d = descrever_contorno(contorno)
        if not (razao_aspecto[0] <= d["razao_aspecto"] <= razao_aspecto[1]):
            motivos["aspecto"] += 1
            continue
        if d["extensao"] < extensao_minima:
            motivos["extensao"] += 1
            continue
        if d["solidez"] < solidez_minima:
            motivos["solidez"] += 1
            continue
        aceitos.append(contorno)
        descritores.append(d)

    return aceitos, descritores, {
        "contornos_brutos": len(brutos), "aceitos": len(aceitos),
        "descartados": len(brutos) - len(aceitos), "por_motivo": motivos,
    }

### 3.9. Bordas (Sobel e Canny) — evidência visual, fora da contagem

O Sobel aproxima o gradiente da imagem por convolução com dois kernels separáveis; o Canny
acrescenta supressão de não-máximos e histerese, devolvendo bordas finas de um pixel.

**Estes operadores não alimentam `findContours` neste pipeline, e essa é uma decisão
deliberada.** Uma borda fechada de um pixel tem dois lados: `findContours` extrai um contorno
externo e outro interno para o mesmo objeto e **duplica a contagem** — o terceiro erro
listado na orientação da AED. A extração de contornos parte sempre da máscara morfológica
preenchida da Etapa E, que é uma região sólida, não uma linha.

O Canny permanece no notebook como evidência visual da estrutura da cena e como material de
comparação para o relatório técnico.

In [ ]:
def bordas_sobel(rgb: np.ndarray, k: int = 3) -> np.ndarray:
    """Magnitude do gradiente por Sobel, normalizada para uint8."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(cinza, cv2.CV_32F, 1, 0, ksize=impar(k))
    gy = cv2.Sobel(cinza, cv2.CV_32F, 0, 1, ksize=impar(k))
    magnitude = cv2.magnitude(gx, gy)
    return cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def bordas_canny(rgb: np.ndarray, sigma: float = 0.33) -> np.ndarray:
    """Canny com histerese ancorada na mediana da imagem (sem constantes magicas)."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mediana = float(np.median(cinza))
    inferior = int(max(0, (1.0 - sigma) * mediana))
    superior = int(min(255, (1.0 + sigma) * mediana))
    return cv2.Canny(cinza, inferior, superior)

---
## 4. Orquestrador e visualização

`executar_pipeline` encadeia as funções da Seção 3 e devolve **todos os estágios
intermediários**, não apenas o resultado final. É isso que permite montar os painéis
"antes e depois" exigidos como evidência visual sem reexecutar o processamento.

In [ ]:
def executar_pipeline(entrada, params: Parametros) -> dict:
    """Executa o pipeline completo sobre uma imagem (caminho ou array RGB).

    Devolve um dicionario com as etapas intermediarias, os contornos aceitos,
    os descritores de cada objeto, o limiar aplicado e estatisticas de filtragem.
    """
    rgb = carregar_imagem(entrada) if isinstance(entrada, (str, Path)) else entrada
    etapas = {}

    imagem, escala = redimensionar(rgb, params.largura_trabalho)
    etapas["1_original"] = imagem

    etapas["2_iluminacao"] = (corrigir_iluminacao_clahe(imagem, params.clahe_clip, params.clahe_grade)
                              if params.usar_clahe else imagem)
    etapas["3_suavizacao"] = suavizar(etapas["2_iluminacao"], params.suavizacao, params.suavizacao_k)
    etapas["4_evidencia"] = mapa_evidencia_cromatica(etapas["3_suavizacao"], params.faixas)

    mascara, limiar = limiarizar(etapas["4_evidencia"], params.metodo_limiar,
                                 params.limiar_global, params.adapt_bloco, params.adapt_c)
    etapas["5_binaria"] = mascara
    etapas["6_morfologia"] = limpar_mascara(mascara, params.k_abertura, params.k_fechamento,
                                            params.preencher_buracos)

    contornos, objetos, stats = extrair_contornos(
        etapas["6_morfologia"], params.area_minima, params.area_maxima,
        params.razao_aspecto, params.extensao_minima, params.solidez_minima)

    return {"etapas": etapas, "contornos": contornos, "objetos": objetos,
            "limiar": limiar, "escala": escala, "estatisticas": stats,
            "dimensoes_trabalho": imagem.shape[:2]}


def recortar_regioes(resultado: dict, margem: float = 0.12) -> list[np.ndarray]:
    """Recorta a ROI normalizada de cada objeto — entrada da etapa de IA da N2."""
    imagem = resultado["etapas"]["1_original"]
    altura, largura = imagem.shape[:2]
    recortes = []
    for obj in resultado["objetos"]:
        mx, my = int(obj["w"] * margem), int(obj["h"] * margem)
        x0, y0 = max(0, obj["x"] - mx), max(0, obj["y"] - my)
        x1, y1 = min(largura, obj["x"] + obj["w"] + mx), min(altura, obj["y"] + obj["h"] + my)
        if x1 > x0 and y1 > y0:
            recortes.append(imagem[y0:y1, x0:x1])
    return recortes

In [ ]:
CORES_FORMA = {
    "circular": (231, 76, 60), "octogonal": (192, 57, 43),
    "triangular": (243, 156, 18), "triangular_invertido": (211, 84, 0),
    "losango": (241, 196, 15), "retangular": (41, 128, 185),
    "quadrilatero": (127, 140, 141), "indefinido": (149, 165, 166),
}


def desenhar_deteccoes(resultado: dict, espessura: int = 2) -> np.ndarray:
    """Sobrepoe contorno, caixa, centroide e rotulo de forma sobre a imagem original."""
    tela = resultado["etapas"]["1_original"].copy()
    for i, (contorno, obj) in enumerate(zip(resultado["contornos"], resultado["objetos"]), 1):
        cor = CORES_FORMA.get(obj["forma"], (149, 165, 166))
        cv2.drawContours(tela, [contorno], -1, cor, espessura)
        cv2.rectangle(tela, (obj["x"], obj["y"]),
                      (obj["x"] + obj["w"], obj["y"] + obj["h"]), cor, 1)
        cv2.circle(tela, (int(obj["cx"]), int(obj["cy"])), 3, (255, 255, 255), -1)
        cv2.circle(tela, (int(obj["cx"]), int(obj["cy"])), 3, cor, 1)
        rotulo = f"{i} {obj['forma']}{'?' if obj['forma_ambigua'] else ''}"
        y_texto = max(12, obj["y"] - 6)
        cv2.putText(tela, rotulo, (obj["x"], y_texto),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(tela, rotulo, (obj["x"], y_texto),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, cor, 1, cv2.LINE_AA)
    return tela


TITULOS_ETAPAS = {
    "1_original": "1. Entrada redimensionada",
    "2_iluminacao": "2. Correção de iluminação (CLAHE em L*)",
    "3_suavizacao": "3. Suavização (Gaussiano)",
    "4_evidencia": "4. Mapa de evidência cromática (HSV)",
    "5_binaria": "5. Limiarização",
    "6_morfologia": "6. Morfologia + preenchimento",
}


def painel_pipeline(resultado: dict, titulo: str = "", salvar: Path | None = None):
    """Grade 2x4 com as seis etapas, o Canny (fora da contagem) e o resultado final."""
    etapas = resultado["etapas"]
    canny = bordas_canny(etapas["3_suavizacao"])
    quadros = [(TITULOS_ETAPAS[k], etapas[k]) for k in TITULOS_ETAPAS]
    quadros.append(("7. Canny — evidência visual,\nnão alimenta findContours", canny))
    n_obj = len(resultado["objetos"])
    quadros.append((f"8. Saída: {n_obj} objeto(s) detectado(s)", desenhar_deteccoes(resultado)))

    fig, eixos = plt.subplots(2, 4, figsize=(16, 7.2))
    for eixo, (nome, imagem) in zip(eixos.ravel(), quadros):
        if imagem.ndim == 2:
            eixo.imshow(imagem, cmap="magma" if "evidência" in nome else "gray", vmin=0, vmax=255)
        else:
            eixo.imshow(imagem)
        eixo.set_title(nome, fontsize=9)
        eixo.axis("off")
    limiar = resultado["limiar"]
    texto_limiar = "local" if np.isnan(limiar) else f"{limiar:.0f}"
    fig.suptitle(f"{titulo}   ·   limiar aplicado: {texto_limiar}   ·   "
                 f"contornos brutos: {resultado['estatisticas']['contornos_brutos']} → "
                 f"aceitos: {resultado['estatisticas']['aceitos']}",
                 fontsize=11, y=1.0)
    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight")
    return fig

---
## 5. Calibração dos parâmetros

Os parâmetros que a orientação da AED manda registrar — **limiar, tamanho de kernel e área
mínima** — são obtidos aqui por medição. O procedimento é dividido em duas partes, porque os
parâmetros são de naturezas diferentes:

- **5.1 — Os kernels morfológicos** saem diretamente da *escala física do objeto* medida na
  Seção 2.1. Não dependem de desempenho: são consequência geométrica do tamanho da placa.
- **5.2 — A área mínima e o método de limiarização** governam um *compromisso entre precisão
  e recall*. Não existe valor "correto" derivável da geometria; existe uma curva de
  trade-off, que é medida e da qual se escolhe um ponto por critério declarado.

### Protocolo anti-viés

A escolha de 5.2 usa uma **amostra de ajuste**; as métricas finalmente reportadas na Seção 8
vêm de uma **amostra de validação disjunta**, que não participa de nenhuma decisão. Sem essa
separação, o desempenho publicado seria otimista por construção — o parâmetro teria sido
escolhido no mesmo conjunto em que é avaliado.

### 5.1. Kernels morfológicos — derivados da escala do objeto

| Parâmetro | Regra | Justificativa |
|---|---|---|
| `k_abertura` | `ímpar(0,10 × p10 do lado equivalente)` | Precisa apagar ruído sem apagar a **menor** placa detectável; por isso ancora no percentil 10, não na mediana. |
| `k_fechamento` | `ímpar(0,25 × mediana do lado equivalente)` | Precisa vencer a espessura do miolo branco de uma placa **típica**, para que as duas margens da orla se toquem. |

A regra degrada para valores padrão seguros caso o dataset venha sem anotações.

In [ ]:
def area_minima_para_quantil(escala: pd.DataFrame, quantil: float,
                             fator_preenchimento: float = 0.45,
                             minimo: int = 30) -> int:
    """Area minima que descarta o quantil inferior das placas anotadas.

    O fator 0.45 converte area da *caixa anotada* em area da *figura inscrita*:
    um triangulo e um losango preenchem cerca de 0.50 da caixa; 0.45 da margem.
    """
    if escala is None or escala.empty:
        return 200
    return int(max(minimo, round(float(escala["area_px"].quantile(quantil)) * fator_preenchimento)))


def area_maxima_para_quantil(escala: pd.DataFrame, quantil: float) -> int:
    """Teto de area: descarta o quantil SUPERIOR das placas anotadas. `1.0` = sem teto.

    O teto existe para rejeitar ceu, fachadas e maciços de vegetacao, que compartilham
    matiz e saturacao com as placas mas sao ordens de grandeza maiores que elas.
    """
    if escala is None or escala.empty or quantil >= 1.0:
        return 0
    return int(round(float(escala["area_px"].quantile(quantil))))


def calibrar_kernels(escala: pd.DataFrame, base: Parametros | None = None) -> tuple[Parametros, dict]:
    """Deriva os kernels morfologicos da distribuicao de tamanho das placas anotadas."""
    params = base or Parametros()
    if escala is None or escala.empty:
        return params, {"origem": "padrao (dataset sem anotacoes)"}

    lado_p10 = float(escala["lado_equivalente_px"].quantile(0.10))
    lado_mediano = float(escala["lado_equivalente_px"].median())

    params.k_abertura = impar(0.10 * lado_p10, minimo=3)
    params.k_fechamento = impar(0.25 * lado_mediano, minimo=3)

    memoria = {
        "origem": f"{len(escala)} caixas anotadas",
        "lado_equivalente_p10_px": round(lado_p10, 1),
        "lado_equivalente_mediano_px": round(lado_mediano, 1),
        "regra_k_abertura": f"impar(0.10 x {lado_p10:.1f}) = {params.k_abertura}",
        "regra_k_fechamento": f"impar(0.25 x {lado_mediano:.1f}) = {params.k_fechamento}",
    }
    return params, memoria


PARAMS, MEMORIA_CALIBRACAO = calibrar_kernels(ESCALA)

print("KERNELS DERIVADOS DA ESCALA DO OBJETO")
print("-" * 58)
for chave, valor in MEMORIA_CALIBRACAO.items():
    print(f"{chave:<32} {valor}")
print("-" * 58)

### 5.2. Área mínima e método de limiarização — escolhidos por métrica

#### Como as detecções são avaliadas

Cada objeto detectado é casado com as caixas anotadas por `IoU ≥ 0,30`. O limiar é frouxo de
propósito: o objetivo de um detector clássico aqui é *localizar para recortar*, não delimitar
com a precisão de um detector treinado. Do casamento saem `precisão`, `recall` e `F1`.

In [ ]:
def iou(caixa_a: tuple, caixa_b: tuple) -> float:
    """Intersecao sobre uniao de duas caixas no formato (x, y, w, h)."""
    ax, ay, aw, ah = caixa_a
    bx, by, bw, bh = caixa_b
    x0, y0 = max(ax, bx), max(ay, by)
    x1, y1 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    intersecao = max(0, x1 - x0) * max(0, y1 - y0)
    uniao = aw * ah + bw * bh - intersecao
    return intersecao / uniao if uniao > 0 else 0.0


def caixas_anotadas(img: Path, largura: int, altura: int) -> list[tuple]:
    """Converte as anotacoes YOLO normalizadas para pixels na resolucao de trabalho."""
    caixas = []
    for _, cx, cy, w, h in ler_rotulos(img):
        caixas.append((int((cx - w / 2) * largura), int((cy - h / 2) * altura),
                       int(w * largura), int(h * altura)))
    return caixas


def casar_deteccoes(detectadas: list, anotadas: list, limiar_iou: float = 0.30) -> tuple[int, int, int]:
    """Casamento guloso por IoU decrescente. Devolve (verdadeiros_pos, falsos_pos, falsos_neg)."""
    pares = sorted(((iou(d, a), i, j) for i, d in enumerate(detectadas)
                    for j, a in enumerate(anotadas)), reverse=True)
    usados_det, usados_anot = set(), set()
    verdadeiros = 0
    for valor, i, j in pares:
        if valor < limiar_iou:
            break
        if i in usados_det or j in usados_anot:
            continue
        usados_det.add(i)
        usados_anot.add(j)
        verdadeiros += 1
    return verdadeiros, len(detectadas) - verdadeiros, len(anotadas) - verdadeiros


def avaliar_configuracao(caminhos: list[Path], params: Parametros,
                         limiar_iou: float = 0.30) -> dict:
    """Roda o pipeline em varias imagens e agrega precisao, recall e F1."""
    vp = fp = fn = 0
    limiares, tempos = [], []
    for caminho in caminhos:
        inicio = time.perf_counter()
        resultado = executar_pipeline(caminho, params)
        tempos.append((time.perf_counter() - inicio) * 1000)
        altura, largura = resultado["dimensoes_trabalho"]
        detectadas = [(o["x"], o["y"], o["w"], o["h"]) for o in resultado["objetos"]]
        a, b, c = casar_deteccoes(detectadas, caixas_anotadas(caminho, largura, altura), limiar_iou)
        vp, fp, fn = vp + a, fp + b, fn + c
        if not np.isnan(resultado["limiar"]):
            limiares.append(resultado["limiar"])

    precisao = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0
    return {"precisao": round(precisao, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "vp": vp, "fp": fp, "fn": fn,
            "limiar_medio": round(float(np.mean(limiares)), 1) if limiares else float("nan"),
            "limiar_desvio": round(float(np.std(limiares)), 1) if limiares else float("nan"),
            "ms_por_imagem": round(float(np.mean(tempos)), 1)}


def com_parametros(base: Parametros, **alteracoes) -> Parametros:
    """Copia `base` aplicando alteracoes — evita mutar a configuracao corrente."""
    novo = Parametros(**{**base.para_dict(), **alteracoes})
    novo.razao_aspecto = tuple(novo.razao_aspecto)
    return novo

In [ ]:
# Duas amostras DISJUNTAS, sorteadas de uma vez com semente fixa.
anotadas = INVENTARIO[INVENTARIO["n_objetos"] > 0]
_sorteio = anotadas.sample(min(2 * N_AMOSTRA_VALIDACAO, len(anotadas)), random_state=SEMENTE)
_caminhos = [Path(p) for p in _sorteio["caminho"]]
_meio = len(_caminhos) // 2

AMOSTRA_AJUSTE = _caminhos[:_meio]        # escolhe parametros
AMOSTRA_VALIDACAO = _caminhos[_meio:]     # so mede, nunca decide

assert not (set(AMOSTRA_AJUSTE) & set(AMOSTRA_VALIDACAO)), "as amostras precisam ser disjuntas"
print(f"Amostra de ajuste    : {len(AMOSTRA_AJUSTE)} imagens")
print(f"Amostra de validação : {len(AMOSTRA_VALIDACAO)} imagens (disjunta, semente {SEMENTE})")

#### A busca em grade

Cada candidato a área mínima é expresso como **"descartar o quantil inferior `q` das placas
anotadas"** — e não como um número solto de pixels. Assim o parâmetro tem significado:
`q = 0,25` quer dizer "o pipeline abre mão do quartil de placas mais distantes, cuja área é
comparável à do ruído cromático residual".

A busca tem **dois estágios** (busca por coordenadas), o que a mantém barata sem perder o
essencial:

1. **Estágio 1** — quantis inferiores × os quatro métodos de limiarização.
2. **Estágio 2** — fixado o vencedor, varre o **teto de área**, que rejeita céu, fachadas e
   maciços de vegetação. O teto é expresso do mesmo jeito: *"descartar o quantil superior das
   placas anotadas"*, com `1,00` significando "sem teto".

Ambos os estágios decidem **na amostra de ajuste**.

#### Por que existe um teto, e por que ele não resolve tudo

O teto nasceu de uma falha concreta. Na primeira execução sobre o dataset real, uma imagem de
rodovia produziu dois "objetos detectados" e **nenhum deles era a placa**: o maior era o
**céu**, ocupando 13,6% da imagem (31.355 px²). Ele passava pelo piso de área, pela razão de
aspecto (2,74, logo abaixo do limite de 2,85), pela extensão (0,56) e pela solidez (0,80).
Céu azul limpo tem a matiz normativa do azul de indicação, saturação alta e forma convexa —
**filtros de forma não o distinguem de uma placa; só a escala distingue**.

Mas o resultado da varredura merece leitura honesta: o teto que **maximiza F1** é largo o
bastante para esse blob ainda passar. Apertá-lo até removê-lo custa recall nas placas
fotografadas de perto, que existem neste dataset, e o F1 pune essa perda tanto quanto pune o
falso positivo. A tabela abaixo expõe o compromisso — precisão sobe monotonicamente conforme
o teto aperta, enquanto o F1 é praticamente plano.

A consequência prática fica registrada: **quem priorizar precisão — o caso de uso de
inventário viário, em que contar placas a mais é pior que perder uma — deve apertar
`quantil_area_max`**, e o custo dessa escolha está medido, não suposto. O notebook mantém o
critério declarado de antemão (maior F1) em vez de trocá-lo depois de ver o resultado.

In [ ]:
QUANTIS_AREA_MIN = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
QUANTIS_AREA_MAX = [0.90, 0.95, 0.99, 1.00]      # 1.00 = sem teto
METODOS_LIMIAR = ["global", "otsu", "otsu_restrito", "adaptativa"]

# --- Estagio 1: metodo de limiarizacao x piso de area (ainda sem teto) -------
linhas = []
for quantil in QUANTIS_AREA_MIN:
    area = area_minima_para_quantil(ESCALA, quantil)
    for metodo in METODOS_LIMIAR:
        p = com_parametros(PARAMS, area_minima=area, area_maxima=0, metodo_limiar=metodo)
        linhas.append({"estagio": 1, "quantil_area_min": quantil, "area_minima": area,
                       "quantil_area_max": 1.00, "area_maxima": 0, "metodo": metodo,
                       **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

GRADE_1 = pd.DataFrame(linhas)
melhor_1 = GRADE_1.loc[GRADE_1["f1"].idxmax()]
METODO_ESCOLHIDO = str(melhor_1["metodo"])
QUANTIL_ESCOLHIDO = float(melhor_1["quantil_area_min"])
AREA_ESCOLHIDA = int(melhor_1["area_minima"])

print(f"ESTÁGIO 1 — {len(GRADE_1)} configurações × {len(AMOSTRA_AJUSTE)} imagens")
print(GRADE_1.pivot(index="quantil_area_min", columns="metodo", values="f1").round(3).to_string())
print(f"\n  vencedor: {METODO_ESCOLHIDO} | quantil inferior {QUANTIL_ESCOLHIDO:.2f} "
      f"→ area_minima = {AREA_ESCOLHIDA} px² | F1 = {melhor_1['f1']:.3f}")

# --- Estagio 2: teto de area, com o vencedor do estagio 1 fixado -------------
linhas = []
for quantil_max in QUANTIS_AREA_MAX:
    teto = area_maxima_para_quantil(ESCALA, quantil_max)
    p = com_parametros(PARAMS, metodo_limiar=METODO_ESCOLHIDO,
                       area_minima=AREA_ESCOLHIDA, area_maxima=teto)
    linhas.append({"estagio": 2, "quantil_area_min": QUANTIL_ESCOLHIDO,
                   "area_minima": AREA_ESCOLHIDA, "quantil_area_max": quantil_max,
                   "area_maxima": teto, "metodo": METODO_ESCOLHIDO,
                   **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

GRADE_2 = pd.DataFrame(linhas)
melhor_2 = GRADE_2.loc[GRADE_2["f1"].idxmax()]
QUANTIL_MAX_ESCOLHIDO = float(melhor_2["quantil_area_max"])
AREA_MAXIMA_ESCOLHIDA = int(melhor_2["area_maxima"])
F1_AJUSTE = float(melhor_2["f1"])

print(f"\nESTÁGIO 2 — teto de área (método e piso fixados)")
print(GRADE_2[["quantil_area_max", "area_maxima", "precisao", "recall", "f1", "vp", "fp", "fn"]]
      .to_string(index=False))

PARAMS = com_parametros(PARAMS, metodo_limiar=METODO_ESCOLHIDO,
                        area_minima=AREA_ESCOLHIDA, area_maxima=AREA_MAXIMA_ESCOLHIDA)

GRADE = pd.concat([GRADE_1, GRADE_2], ignore_index=True)
teto_txt = "sem teto" if AREA_MAXIMA_ESCOLHIDA == 0 else f"{AREA_MAXIMA_ESCOLHIDA} px²"
print(f"\nADOTADO: {METODO_ESCOLHIDO} | área ∈ [{AREA_ESCOLHIDA} px², {teto_txt}] "
      f"| F1 (ajuste) = {F1_AJUSTE:.3f}")

# Melhor configuracao de cada metodo no estagio 1, para a tabela do relatorio.
COMPARACAO_LIMIAR = (GRADE_1.sort_values("f1", ascending=False)
                     .drop_duplicates("metodo").set_index("metodo"))
COMPARACAO_LIMIAR

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 4))

# (a) Estagio 1: trade-off do piso de area, para o metodo vencedor.
curva = GRADE_1[GRADE_1["metodo"] == METODO_ESCOLHIDO].sort_values("area_minima")
eixos[0].plot(curva["area_minima"], curva["precisao"], "o-", color="#2d6a9f", label="precisão")
eixos[0].plot(curva["area_minima"], curva["recall"], "s-", color="#e08214", label="recall")
eixos[0].plot(curva["area_minima"], curva["f1"], "^-", color="#c0392b", lw=2.2, label="F1")
eixos[0].axvline(AREA_ESCOLHIDA, color="#4d9078", ls="--", lw=2,
                 label=f"adotado: {AREA_ESCOLHIDA} px²")
eixos[0].set_xscale("log")
eixos[0].set_xlabel("piso de área do contorno (px², escala log)")
eixos[0].set_title(f"(a) Estágio 1 — piso de área · {METODO_ESCOLHIDO}")
eixos[0].legend(fontsize=8); eixos[0].grid(alpha=0.3)

# (b) Estagio 2: efeito do teto de area — e aqui que o ceu e removido.
rotulos = [("sem teto" if q >= 1.0 else f"q={q:.2f}") for q in GRADE_2["quantil_area_max"]]
x = np.arange(len(rotulos))
eixos[1].plot(x, GRADE_2["precisao"], "o-", color="#2d6a9f", label="precisão")
eixos[1].plot(x, GRADE_2["recall"], "s-", color="#e08214", label="recall")
eixos[1].plot(x, GRADE_2["f1"], "^-", color="#c0392b", lw=2.2, label="F1")
eixos[1].set_xticks(x); eixos[1].set_xticklabels(rotulos, fontsize=8)
eixos[1].set_xlabel("teto de área (quantil superior das placas anotadas)")
eixos[1].set_title("(b) Estágio 2 — teto de área (céu, fachadas)")
eixos[1].legend(fontsize=8); eixos[1].grid(alpha=0.3)

# (c) Comparacao entre metodos de limiarizacao.
COMPARACAO_LIMIAR[["precisao", "recall", "f1"]].plot(
    kind="bar", ax=eixos[2], color=["#2d6a9f", "#e08214", "#4d9078"], rot=15, width=0.78)
eixos[2].set_title("(c) Melhor configuração de cada método")
eixos[2].set_ylabel("métrica"); eixos[2].legend(fontsize=8); eixos[2].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "01_escolha_de_parametros.png", bbox_inches="tight")
plt.show()

---
## 6. Ablação do pré-processamento

Com os parâmetros já fixados, mede-se o efeito de **desligar** o CLAHE e a suavização.

A ablação responde com número, e não com opinião, ao primeiro erro apontado na orientação da
AED — *"segmentar sem suavizar antes, e concluir que Otsu não funciona"*. Continua rodando
sobre a amostra de ajuste: é diagnóstico, não resultado.

**Como ler a tabela.** Duas leituras separadas, porque as duas etapas se comportam de forma
diferente:

- **O CLAHE se paga.** Compare os pares de mesma suavização: ligar a correção de iluminação
  melhora o F1 em todos eles. Faz sentido — contraluz e sombra deprimem o canal `V`, e é
  exatamente esse canal que porteia o mapa de evidência cromática.
- **A suavização, neste dataset, não.** Desligá-la sai ligeiramente à frente. A causa é a
  origem das imagens: o export do Roboflow já passou por redimensionamento e recompressão
  JPEG, que atenuam o ruído de sensor que o filtro removeria; o que resta do gaussiano é
  erodir as placas menores. O filtro permanece no pipeline por duas razões declaradas — é
  etapa exigida pelo Checkpoint 1, e as imagens de captura autoral da equipe chegam sem esse
  pré-tratamento. **A conclusão registrada é a da medição, não a da expectativa**; quem
  processar imagens brutas deve refazer esta ablação antes de decidir.

In [ ]:
linhas = []
for usar_clahe in (True, False):
    for suavizacao in ("gaussiano", "mediana", "nenhuma"):
        p = com_parametros(PARAMS, usar_clahe=usar_clahe, suavizacao=suavizacao)
        linhas.append({"clahe": usar_clahe, "suavizacao": suavizacao,
                       **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

ABLACAO = pd.DataFrame(linhas).sort_values("f1", ascending=False).reset_index(drop=True)
print(f"ABLAÇÃO — método '{METODO_ESCOLHIDO}', área mínima {PARAMS.area_minima} px²\n")
print(ABLACAO.to_string(index=False))

fig, eixo = plt.subplots(figsize=(7.5, 3.6))
rotulos = [f"CLAHE={'sim' if r.clahe else 'não'}\n{r.suavizacao}" for r in ABLACAO.itertuples()]
cores = ["#4d9078" if (r.clahe == PARAMS.usar_clahe and r.suavizacao == PARAMS.suavizacao)
         else "#2d6a9f" for r in ABLACAO.itertuples()]
eixo.bar(rotulos, ABLACAO["f1"], color=cores)
eixo.set_title("Ablação do pré-processamento (F1) — verde: configuração adotada")
eixo.set_ylabel("F1"); eixo.tick_params(axis="x", labelsize=7.5); eixo.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(DIR_FIGURAS / "02_ablacao_preprocessamento.png", bbox_inches="tight")
plt.show()

### 6.1. O histograma que justifica a limiarização

O gráfico abaixo mostra, para uma imagem representativa, o histograma do mapa de evidência
cromática e onde cada método posiciona seu corte. Ele torna visível a patologia descrita na
Seção 3.6: o pico esmagador em zero — o fundo da cena — puxa o limiar do Otsu clássico para
baixo, enquanto o Otsu restrito, que ignora esses pixels, corta na região onde de fato existe
separação entre placa e resto.

In [ ]:
def diagramar_histograma(caminho: Path, params: Parametros, salvar: Path | None = None):
    """Histograma do mapa de evidencia com os limiares de cada metodo sobrepostos."""
    resultado = executar_pipeline(caminho, params)
    mapa = resultado["etapas"]["4_evidencia"]
    limiares = {m: limiarizar(mapa, m, params.limiar_global)[1]
                for m in ("global", "otsu", "otsu_restrito")}

    fig, eixos = plt.subplots(1, 3, figsize=(14, 3.6))
    eixos[0].imshow(resultado["etapas"]["1_original"]); eixos[0].axis("off")
    eixos[0].set_title("Imagem de referência")
    eixos[1].imshow(mapa, cmap="magma", vmin=0, vmax=255); eixos[1].axis("off")
    eixos[1].set_title("Mapa de evidência cromática")

    eixos[2].hist(mapa.ravel(), bins=64, range=(0, 255), color="#7f8c8d", log=True)
    cores = {"global": "#2d6a9f", "otsu": "#e08214", "otsu_restrito": "#c0392b"}
    for nome, valor in limiares.items():
        if not np.isnan(valor):
            eixos[2].axvline(valor, color=cores[nome], lw=2, ls="--",
                             label=f"{nome} = {valor:.0f}")
    eixos[2].set_title("Histograma do mapa (escala log)")
    eixos[2].set_xlabel("evidência cromática"); eixos[2].set_ylabel("pixels")
    eixos[2].legend(fontsize=8)

    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight")
    return fig


diagramar_histograma(AMOSTRA_AJUSTE[0], PARAMS, DIR_FIGURAS / "03_histograma_limiares.png")
plt.show()

---
## 7. Evidências visuais — antes e depois de cada etapa

> *"Um pipeline validado em uma única imagem não é um pipeline."*

As imagens abaixo são sorteadas com semente fixa entre as que possuem anotação, sem seleção
manual dos casos favoráveis. Cada painel percorre as seis etapas do pipeline, exibe o Canny
(que **não** alimenta a contagem) e fecha com as detecções sobre a imagem de entrada.

In [ ]:
AMOSTRA_EVIDENCIAS = [Path(p) for p in anotadas.sample(
    min(N_EVIDENCIAS_VISUAIS, len(anotadas)), random_state=SEMENTE + 1)["caminho"]]

RESULTADOS_EVIDENCIA = []
for i, caminho in enumerate(AMOSTRA_EVIDENCIAS, 1):
    resultado = executar_pipeline(caminho, PARAMS)
    RESULTADOS_EVIDENCIA.append((caminho, resultado))
    painel_pipeline(resultado, titulo=f"[{i}/{len(AMOSTRA_EVIDENCIAS)}] {caminho.name}",
                    salvar=DIR_FIGURAS / f"04_pipeline_{i:02d}_{caminho.stem[:40]}.png")
    plt.show()

In [ ]:
def mosaico_deteccoes(resultados: list, salvar: Path | None = None):
    """Grade com o resultado final de todas as imagens de evidencia lado a lado."""
    n = len(resultados)
    colunas = min(3, n)
    linhas = math.ceil(n / colunas)
    fig, eixos = plt.subplots(linhas, colunas, figsize=(5.2 * colunas, 3.9 * linhas))
    eixos = np.atleast_1d(eixos).ravel()
    for eixo, (caminho, resultado) in zip(eixos, resultados):
        eixo.imshow(desenhar_deteccoes(resultado))
        eixo.set_title(f"{caminho.name[:34]}\ndetectados: {len(resultado['objetos'])}  ·  "
                       f"anotados: {len(ler_rotulos(caminho))}", fontsize=8)
        eixo.axis("off")
    for eixo in eixos[n:]:
        eixo.axis("off")
    fig.suptitle("Saída do pipeline — contagem e classe geométrica por objeto", y=1.0)
    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight")
    return fig


mosaico_deteccoes(RESULTADOS_EVIDENCIA, DIR_FIGURAS / "05_mosaico_deteccoes.png")
plt.show()

In [ ]:
# Tabela de descritores: a saida numerica que responde ao problema.
linhas = []
for caminho, resultado in RESULTADOS_EVIDENCIA:
    for i, obj in enumerate(resultado["objetos"], 1):
        linhas.append({"imagem": caminho.name, "objeto": i, "forma": obj["forma"],
                       "ambigua": obj["forma_ambigua"],
                       "area_px2": round(obj["area_px"]),
                       "centroide_x": round(obj["cx"], 1), "centroide_y": round(obj["cy"], 1),
                       "largura": obj["w"], "altura": obj["h"],
                       "circularidade": round(obj["circularidade"], 3),
                       "solidez": round(obj["solidez"], 3),
                       "extensao": round(obj["extensao"], 3)})

DESCRITORES = pd.DataFrame(linhas)
DESCRITORES.to_csv(DIR_SAIDA / "descritores_objetos.csv", index=False, encoding="utf-8")
print(f"{len(DESCRITORES)} objetos descritos em {len(RESULTADOS_EVIDENCIA)} imagens\n")
DESCRITORES

In [ ]:
# Regioes de interesse normalizadas: e este recorte que alimenta a CNN da 2a Etapa.
recortes = [r for _, resultado in RESULTADOS_EVIDENCIA for r in recortar_regioes(resultado)][:12]
if recortes:
    colunas = min(6, len(recortes))
    linhas_grade = math.ceil(len(recortes) / colunas)
    fig, eixos = plt.subplots(linhas_grade, colunas, figsize=(1.9 * colunas, 2.1 * linhas_grade))
    eixos = np.atleast_1d(eixos).ravel()
    for eixo, recorte in zip(eixos, recortes):
        eixo.imshow(cv2.resize(recorte, (96, 96), interpolation=cv2.INTER_CUBIC))
        eixo.axis("off")
    for eixo in eixos[len(recortes):]:
        eixo.axis("off")
    fig.suptitle("ROIs extraídas e normalizadas — entrada da etapa de IA (N2)", y=1.01)
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "06_rois_normalizadas.png", bbox_inches="tight")
    plt.show()
else:
    print("Nenhuma ROI extraida nesta amostra.")

---
## 8. Avaliação quantitativa — amostra de validação retida

Esta seção mede o desempenho sobre a **amostra de validação**, disjunta da amostra de ajuste
usada na Seção 5.2. Nenhuma imagem daqui participou da escolha de qualquer parâmetro, então o
número abaixo é uma estimativa honesta — e não a nota de prova de quem viu o gabarito.

Duas leituras complementares:

- **Nível de detecção** — `precisão`, `recall` e `F1` por casamento `IoU ≥ 0,30`. Mede se as
  placas foram *localizadas*.
- **Nível de contagem** — erro absoluto médio entre o número de objetos detectados e o número
  de objetos anotados. É a métrica que interessa ao caso de uso de inventário viário.

O resultado esperado nesta etapa **não é alto**, e isso é informação, não fracasso: um
detector puramente cromático é a linha de base contra a qual o detector treinado da 2ª Etapa
será comparado. O valor do número está em existir e ser reprodutível.

In [ ]:
def avaliar_em_detalhe(caminhos: list[Path], params: Parametros,
                       limiar_iou: float = 0.30) -> pd.DataFrame:
    """Avaliacao imagem a imagem: detectados, anotados, VP/FP/FN e erro de contagem."""
    linhas = []
    for caminho in caminhos:
        resultado = executar_pipeline(caminho, params)
        altura, largura = resultado["dimensoes_trabalho"]
        detectadas = [(o["x"], o["y"], o["w"], o["h"]) for o in resultado["objetos"]]
        anotadas_img = caixas_anotadas(caminho, largura, altura)
        vp, fp, fn = casar_deteccoes(detectadas, anotadas_img, limiar_iou)
        linhas.append({"imagem": caminho.name,
                       "detectados": len(detectadas), "anotados": len(anotadas_img),
                       "vp": vp, "fp": fp, "fn": fn,
                       "erro_contagem": len(detectadas) - len(anotadas_img)})
    return pd.DataFrame(linhas)


AVALIACAO = avaliar_em_detalhe(AMOSTRA_VALIDACAO, PARAMS)

vp, fp, fn = int(AVALIACAO["vp"].sum()), int(AVALIACAO["fp"].sum()), int(AVALIACAO["fn"].sum())
precisao = vp / (vp + fp) if vp + fp else 0.0
recall = vp / (vp + fn) if vp + fn else 0.0
f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0

RESUMO_AVALIACAO = {
    "amostra": "validacao retida (disjunta da amostra de ajuste)",
    "imagens_avaliadas": len(AVALIACAO),
    "objetos_anotados": int(AVALIACAO["anotados"].sum()),
    "objetos_detectados": int(AVALIACAO["detectados"].sum()),
    "verdadeiros_positivos": vp, "falsos_positivos": fp, "falsos_negativos": fn,
    "precisao": round(precisao, 3), "recall": round(recall, 3), "f1": round(f1, 3),
    "limiar_iou": 0.30,
    "erro_absoluto_medio_contagem": round(float(AVALIACAO["erro_contagem"].abs().mean()), 2),
    "imagens_com_contagem_exata": int((AVALIACAO["erro_contagem"] == 0).sum()),
    "f1_na_amostra_de_ajuste": round(F1_AJUSTE, 3),
    "diferenca_ajuste_validacao": round(F1_AJUSTE - f1, 3),
}

print("DESEMPENHO DA LINHA DE BASE CLASSICA (PDI, sem aprendizado)")
print("=" * 62)
for chave, valor in RESUMO_AVALIACAO.items():
    print(f"{chave:<34} {valor}")
print("=" * 62)
print(f"\nF1 na amostra de ajuste     : {F1_AJUSTE:.3f}  (onde os parametros foram escolhidos)")
print(f"F1 na amostra de validacao  : {f1:.3f}  (retida — estimativa honesta)")
if F1_AJUSTE - f1 > 0.05:
    print("\nA diferenca entre as duas mede o otimismo da selecao de parametros: escolher\n"
          "entre varias configuracoes na mesma amostra em que se mede inflaria o resultado.\n"
          "E exatamente esse vies que a amostra retida evita.")
AVALIACAO.head(15)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12.5, 3.6))

eixos[0].bar(["precisão", "recall", "F1"], [precisao, recall, f1],
             color=["#2d6a9f", "#e08214", "#4d9078"])
eixos[0].set_ylim(0, 1); eixos[0].grid(axis="y", alpha=0.3)
eixos[0].set_title(f"Detecção · IoU ≥ 0,30 · {len(AVALIACAO)} imagens retidas")
for i, valor in enumerate([precisao, recall, f1]):
    eixos[0].text(i, valor + 0.02, f"{valor:.2f}", ha="center", fontsize=9)

limite = max(3, int(AVALIACAO["erro_contagem"].abs().max()))
eixos[1].hist(AVALIACAO["erro_contagem"], bins=np.arange(-limite - 0.5, limite + 1.5),
              color="#2d6a9f", edgecolor="white")
eixos[1].axvline(0, color="#c1440e", lw=2, label="contagem exata")
eixos[1].set_title("Erro de contagem (detectados − anotados)")
eixos[1].set_xlabel("erro"); eixos[1].legend(fontsize=8); eixos[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "07_avaliacao_quantitativa.png", bbox_inches="tight")
plt.show()

In [ ]:
# Distribuicao das classes geometricas atribuidas em toda a amostra de validacao.
formas = []
for caminho in AMOSTRA_VALIDACAO:
    for obj in executar_pipeline(caminho, PARAMS)["objetos"]:
        formas.append({"forma": obj["forma"], "ambigua": obj["forma_ambigua"]})

if formas:
    DISTRIBUICAO_FORMAS = (pd.DataFrame(formas)
                           .groupby("forma")
                           .agg(objetos=("forma", "size"), ambiguos=("ambigua", "sum"))
                           .sort_values("objetos", ascending=False))
    DISTRIBUICAO_FORMAS["% do total"] = (DISTRIBUICAO_FORMAS["objetos"] /
                                         DISTRIBUICAO_FORMAS["objetos"].sum() * 100).round(1)
    print("CLASSIFICACAO GEOMETRICA NA AMOSTRA DE VALIDACAO\n")
    print(DISTRIBUICAO_FORMAS.to_string())
else:
    DISTRIBUICAO_FORMAS = pd.DataFrame()
    print("Nenhum objeto detectado na amostra.")
DISTRIBUICAO_FORMAS

---
## 9. Diagrama da arquitetura da solução

O fluxo de dados da entrada até a saída pretendida, com o ponto exato em que o modelo de IA
entra nas próximas etapas do projeto.

A fronteira entre as duas etapas é a **ROI normalizada**: o Checkpoint 1 entrega o recorte da
região de interesse com seus descritores geométricos, e é exatamente esse recorte que a 2ª
Etapa consome. As rotinas de correção de iluminação e o filtro por área permanecem como pré e
pós-processamento do detector treinado — o pipeline clássico não é descartado, vira
infraestrutura e linha de base comparativa.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def _caixa(eixo, x, y, largura, altura, texto, cor_fundo, cor_borda,
           tamanho=8.0, estilo="solid", peso="normal"):
    eixo.add_patch(FancyBboxPatch((x, y), largura, altura,
                                  boxstyle="round,pad=0.02,rounding_size=0.12",
                                  linewidth=1.3, edgecolor=cor_borda, facecolor=cor_fundo,
                                  linestyle=estilo, zorder=2))
    eixo.text(x + largura / 2, y + altura / 2, texto, ha="center", va="center",
              fontsize=tamanho, color="#1b262c", zorder=3, fontweight=peso, linespacing=1.35)


def _seta(eixo, origem, destino, estilo="solid", cor="#5a6b7b"):
    eixo.add_patch(FancyArrowPatch(origem, destino, arrowstyle="-|>", mutation_scale=13,
                                   linewidth=1.3, color=cor, linestyle=estilo, zorder=1))


def desenhar_diagrama_arquitetura(params: Parametros, salvar: Path | None = None):
    """Diagrama de fluxo do sistema, da imagem de entrada a saida da 2a Etapa."""
    faixas = [
        ("1 · AQUISIÇÃO", "#eef4f9", "#2d6a9f", [
            f"Dataset Roboflow\n{PROJETO} v{VERSAO}",
            f"Inventário automático\n{len(INVENTARIO)} imagens · {len(CLASSES)} classes",
            "Anotações YOLO\n(escala real do objeto)"]),
        ("2 · PRÉ-PROCESSAMENTO", "#fdf3e7", "#e08214", [
            f"Redimensionar\nlargura = {params.largura_trabalho} px",
            f"CLAHE no canal L* (LAB)\nclip = {params.clahe_clip} · grade {params.clahe_grade}×{params.clahe_grade}",
            f"Suavização {params.suavizacao}\nkernel {params.suavizacao_k}×{params.suavizacao_k}"]),
        ("3 · SEGMENTAÇÃO", "#eef7f2", "#4d9078", [
            "Mapa de evidência\ncromática (HSV, CONTRAN)",
            f"Limiarização\n{params.metodo_limiar}",
            f"Abertura {params.k_abertura}×{params.k_abertura} → fechamento "
            f"{params.k_fechamento}×{params.k_fechamento}\n→ preenchimento"]),
        ("4 · EXTRAÇÃO", "#f4eef7", "#8e5aa8", [
            "findContours\nRETR_EXTERNAL",
            (f"Filtros · área ∈ [{params.area_minima}, "
             f"{params.area_maxima if params.area_maxima else '∞'}] px²\naspecto · extensão · solidez"),
            "Descritores geométricos\ne classificação de forma"]),
        ("5 · SAÍDA — CHECKPOINT 1", "#fcecec", "#c0392b", [
            "Contagem · área\ncentroide · caixa",
            "Classe geométrica\n(circular, losango, ...)",
            "ROI normalizada\n+ CSV de descritores"]),
    ]

    fig, eixo = plt.subplots(figsize=(13.5, 10.6))
    eixo.set_xlim(0, 13.5); eixo.set_ylim(-0.45, 10.25); eixo.axis("off")

    x0, largura_caixa, vao = 2.35, 3.25, 0.35
    altura_caixa, altura_faixa = 0.95, 1.55
    topo = 9.55

    centros_faixa = []
    for i, (titulo, fundo, borda, caixas) in enumerate(faixas):
        y = topo - (i + 1) * altura_faixa
        eixo.add_patch(FancyBboxPatch((0.25, y - 0.12), 13.0, altura_faixa - 0.18,
                                      boxstyle="round,pad=0.02,rounding_size=0.1",
                                      linewidth=0, facecolor=fundo, zorder=0))
        eixo.text(0.45, y + altura_caixa / 2 - 0.12, titulo.replace(" · ", "\n"),
                  fontsize=8.5, fontweight="bold", color=borda, va="center", linespacing=1.5)
        for j, texto in enumerate(caixas):
            x = x0 + j * (largura_caixa + vao)
            _caixa(eixo, x, y, largura_caixa, altura_caixa, texto, "white", borda)
            if j:
                _seta(eixo, (x - vao, y + altura_caixa / 2), (x - 0.02, y + altura_caixa / 2))
        centros_faixa.append((y, y + altura_caixa))
        if i:
            y_ant = centros_faixa[i - 1][0]
            _seta(eixo, (x0 + largura_caixa / 2, y_ant - 0.02),
                  (x0 + largura_caixa / 2, y + altura_caixa + 0.02))

    # Faixa da 2a Etapa: tracejada, indicando o que ainda nao foi implementado.
    y = topo - 6 * altura_faixa - 0.25
    eixo.add_patch(FancyBboxPatch((0.25, y - 0.12), 13.0, altura_faixa - 0.18,
                                  boxstyle="round,pad=0.02,rounding_size=0.1",
                                  linewidth=1.4, linestyle="--", edgecolor="#5a6b7b",
                                  facecolor="#f4f6f7", zorder=0))
    eixo.text(0.45, y + altura_caixa / 2 - 0.12, "6 · 2ª ETAPA\n(N2 — IA)",
              fontsize=8.5, fontweight="bold", color="#34495e", va="center", linespacing=1.5)
    for j, texto in enumerate(["Detector YOLO\ntreinado em cena completa",
                               "CNN classificadora\n(GTSRB · 43 classes)",
                               "Classe da placa + laudo\naval. por mAP, IoU e acurácia"]):
        x = x0 + j * (largura_caixa + vao)
        _caixa(eixo, x, y, largura_caixa, altura_caixa, texto, "white", "#5a6b7b", estilo="--")
        if j:
            _seta(eixo, (x - vao, y + altura_caixa / 2), (x - 0.02, y + altura_caixa / 2), "--")

    x_roi = x0 + 2 * (largura_caixa + vao) + largura_caixa / 2
    _seta(eixo, (x_roi, centros_faixa[-1][0] - 0.02), (x_roi, y + altura_caixa + 0.02), "--", "#c0392b")
    eixo.text(x_roi - 0.18, y + altura_caixa + 0.16,
              "a ROI normalizada é a fronteira entre as duas etapas",
              fontsize=7.5, style="italic", color="#c0392b", va="bottom", ha="right")

    eixo.text(6.85, 9.95, "Sistema Inteligente de Detecção e Classificação de Placas de Trânsito",
              ha="center", fontsize=12, fontweight="bold", color="#1b262c")
    eixo.text(6.85, 9.66, "Arquitetura da solução — pipeline de PDI clássico (N1) e "
                          "extensão por aprendizado profundo (N2)",
              ha="center", fontsize=8.5, color="#5a6b7b")

    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight", facecolor="white")
    return fig


desenhar_diagrama_arquitetura(PARAMS, DIR_DOCS / "arquitetura_pipeline.png")
plt.show()

---
## 10. Registro dos parâmetros e exportação dos artefatos

Tudo o que o repositório precisa é gerado aqui: o registro dos parâmetros adotados (exigido
no README pelo enunciado), as tabelas em CSV, as figuras e um `.zip` com o conjunto.

In [ ]:
REGISTRO = {
    "projeto": "Sistema Inteligente de Deteccao e Classificacao de Placas de Transito",
    "etapa": "AED 2a Etapa - Checkpoint 1 (N1)",
    "disciplina": "CDI1021 - Visao Computacional (2026/2) - PUC Goias",
    "gerado_em": time.strftime("%Y-%m-%d %H:%M:%S"),
    "ambiente": {"python": sys.version.split()[0], "opencv": cv2.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__, "semente": SEMENTE},
    "dataset": {
        "fonte": f"Roboflow Universe / {WORKSPACE}/{PROJETO} versao {VERSAO}",
        "url": f"https://universe.roboflow.com/{WORKSPACE}/{PROJETO}/dataset/{VERSAO}",
        "local": str(DIR_DATASET), "imagens": int(len(INVENTARIO)),
        "objetos_anotados": int(INVENTARIO["n_objetos"].sum()),
        "classes": int(len(CLASSES)),
        "dimensao_mediana": (f"{int(dim['largura'].median())}x{int(dim['altura'].median())}"
                             if len(dim) else "n/d"),
    },
    "parametros_adotados": PARAMS.para_dict(),
    "memoria_de_calculo": MEMORIA_CALIBRACAO,
    "protocolo": {
        "amostra_ajuste": len(AMOSTRA_AJUSTE),
        "amostra_validacao": len(AMOSTRA_VALIDACAO),
        "disjuntas": True,
        "observacao": ("parametros escolhidos na amostra de ajuste; metricas reportadas "
                       "na amostra de validacao, que nao participou de nenhuma decisao"),
    },
    "escolha_de_parametros": {
        "metodo_escolhido": METODO_ESCOLHIDO,
        "quantil_area_min": QUANTIL_ESCOLHIDO,
        "area_minima_px2": AREA_ESCOLHIDA,
        "quantil_area_max": QUANTIL_MAX_ESCOLHIDO,
        "area_maxima_px2": AREA_MAXIMA_ESCOLHIDA,
        "f1_na_amostra_de_ajuste": round(F1_AJUSTE, 3),
        "criterio": (f"busca por coordenadas em 2 estagios, {len(GRADE)} configuracoes sobre "
                     f"{len(AMOSTRA_AJUSTE)} imagens de ajuste, maior F1 com IoU >= 0.30"),
        "grade": json.loads(GRADE.to_json(orient="records")),
        "melhor_por_metodo": json.loads(COMPARACAO_LIMIAR.reset_index().to_json(orient="records")),
    },
    "ablacao_preprocessamento": json.loads(ABLACAO.to_json(orient="records")),
    "desempenho": RESUMO_AVALIACAO,
}

arquivo_json = DIR_SAIDA / "parametros_adotados.json"
arquivo_json.write_text(json.dumps(REGISTRO, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Registro salvo em {arquivo_json}")
print(json.dumps(REGISTRO["parametros_adotados"], indent=2, ensure_ascii=False))

In [ ]:
def tabela_markdown_parametros(registro: dict) -> str:
    """Gera a secao de parametros pronta para colar no README.md do repositorio."""
    p = registro["parametros_adotados"]
    memoria = registro["memoria_de_calculo"]
    escolha = registro["escolha_de_parametros"]
    protocolo = registro["protocolo"]
    d = registro["dataset"]
    desempenho = registro["desempenho"]
    linhas = [
        "## Parâmetros adotados",
        "",
        f"Gerado automaticamente pelo notebook em {registro['gerado_em']} "
        f"(semente {registro['ambiente']['semente']}, OpenCV {registro['ambiente']['opencv']}, "
        f"Python {registro['ambiente']['python']}).",
        "",
        "| Parâmetro | Valor | Origem |",
        "|---|---|---|",
        f"| Largura de trabalho | {p['largura_trabalho']} px | Padroniza a escala em pixels entre imagens |",
        f"| CLAHE (clip / grade) | {p['clahe_clip']} / {p['clahe_grade']}×{p['clahe_grade']} | Canal L* do LAB, preserva a matiz |",
        f"| Suavização | {p['suavizacao']} {p['suavizacao_k']}×{p['suavizacao_k']} | Menor que a placa típica; precede a limiarização |",
        f"| **Kernel de abertura** | **{p['k_abertura']}×{p['k_abertura']}** | `{memoria.get('regra_k_abertura', 'padrão')}` |",
        f"| **Kernel de fechamento** | **{p['k_fechamento']}×{p['k_fechamento']}** | `{memoria.get('regra_k_fechamento', 'padrão')}` |",
        f"| **Método de limiarização** | **{escolha['metodo_escolhido']}** | {escolha['criterio']} |",
        f"| **Área mínima de contorno** | **{escolha['area_minima_px2']} px²** | Descarta o quantil {escolha['quantil_area_min']:.2f} inferior das placas anotadas (× 0,45 de preenchimento) |",
        (f"| **Área máxima de contorno** | **{escolha['area_maxima_px2']} px²** | "
         f"Descarta o quantil superior a {escolha['quantil_area_max']:.2f} — ataca céu, fachadas e vegetação (alcance medido na Seção 5.2) |"
         if escolha["area_maxima_px2"] else
         "| **Área máxima de contorno** | sem teto | A varredura do estágio 2 não indicou ganho em limitar a área |"),
        f"| Razão de aspecto aceita | {p['razao_aspecto'][0]}–{p['razao_aspecto'][1]} | Rejeita postes, faixas e meios-fios |",
        f"| Extensão mínima | {p['extensao_minima']} | Rejeita contornos rendilhados (vegetação) |",
        f"| Solidez mínima | {p['solidez_minima']} | Toda placa normativa é convexa |",
        "",
        "### Dataset",
        "",
        f"- Fonte: {d['fonte']}",
        f"- {d['imagens']} imagens · {d['objetos_anotados']} objetos anotados · {d['classes']} classes",
        f"- Dimensão mediana: {d['dimensao_mediana']} px",
        "",
        "### Protocolo de avaliação",
        "",
        f"- {protocolo['amostra_ajuste']} imagens de **ajuste** (escolha dos parâmetros) e "
        f"{protocolo['amostra_validacao']} de **validação**, disjuntas, sorteadas com semente fixa",
        f"- Casamento detecção ↔ anotação por IoU ≥ {desempenho['limiar_iou']}",
        "",
        "### Desempenho da linha de base clássica (amostra de validação retida)",
        "",
        f"- Precisão {desempenho['precisao']} · Recall {desempenho['recall']} · "
        f"F1 {desempenho['f1']} ({desempenho['imagens_avaliadas']} imagens)",
        f"- Erro absoluto médio de contagem: {desempenho['erro_absoluto_medio_contagem']} objetos por imagem",
        f"- Contagem exata em {desempenho['imagens_com_contagem_exata']} de "
        f"{desempenho['imagens_avaliadas']} imagens",
    ]
    return "\n".join(linhas)


texto_md = tabela_markdown_parametros(REGISTRO)
(DIR_SAIDA / "parametros_adotados.md").write_text(texto_md, encoding="utf-8")
print(texto_md)

In [ ]:
GRADE.to_csv(DIR_SAIDA / "busca_em_grade.csv", index=False, encoding="utf-8")
COMPARACAO_LIMIAR.to_csv(DIR_SAIDA / "comparacao_limiarizacao.csv", encoding="utf-8")
ABLACAO.to_csv(DIR_SAIDA / "ablacao_preprocessamento.csv", index=False, encoding="utf-8")
AVALIACAO.to_csv(DIR_SAIDA / "avaliacao_por_imagem.csv", index=False, encoding="utf-8")
if not DISTRIBUICAO_FORMAS.empty:
    DISTRIBUICAO_FORMAS.to_csv(DIR_SAIDA / "distribuicao_formas.csv", encoding="utf-8")

pacote = RAIZ / "outputs_checkpoint1.zip"
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(DIR_SAIDA.rglob("*")):
        if arquivo.is_file():
            z.write(arquivo, arquivo.relative_to(RAIZ))
    diagrama = DIR_DOCS / "arquitetura_pipeline.png"
    if diagrama.exists():
        z.write(diagrama, diagrama.relative_to(RAIZ))

print("Artefatos gerados em outputs/:")
for arquivo in sorted(DIR_SAIDA.rglob("*")):
    if arquivo.is_file():
        print(f"  {arquivo.relative_to(DIR_SAIDA)}  ({arquivo.stat().st_size / 1024:.0f} KB)")
print(f"\nPacote: {pacote} ({pacote.stat().st_size / 1024 / 1024:.1f} MB)")

if EM_COLAB:
    from google.colab import files
    print("\nBaixe o pacote para versionar no repositorio:")
    files.download(str(pacote))

---
## 11. Limitações identificadas e caminho para a 2ª Etapa

Registro honesto do que este pipeline **não** resolve — é sobre estes pontos que o modelo
treinado da 2ª Etapa precisará ganhar.

### Limitações do pipeline clássico

1. **Falsos positivos de mesma cromaticidade.** Lanternas traseiras, veículos vermelhos,
   toldos e sinalização publicitária compartilham matiz e saturação com as placas de
   regulamentação. Os filtros de forma removem parte deles, mas um objeto vermelho
   aproximadamente convexo e de proporção compatível é indistinguível por cor e geometria.

2. **Placas verdes fora do escopo.** A exclusão do verde (Seção 3.1) deixa as placas de
   indicação verdes sem cobertura. Incluí-las exigiria um discriminante adicional contra
   vegetação — textura, por exemplo — que foge ao escopo do Checkpoint 1.

3. **Círculo e octógono são indistinguíveis sob perspectiva.** Documentado com medições na
   Seção 3.8: a projeção oblíqua achata ambas as formas em elipses de razão equivalente.
   O código marca esses casos com `forma_ambigua = True` em vez de arbitrar.

4. **Placas pequenas e distantes.** Abaixo da área mínima calibrada, o objeto é descartado por
   construção. Baixar o limiar recuperaria essas placas ao custo de admitir ruído na
   contagem — o compromisso é explícito e ajustável em `Parametros.area_minima`.

5. **Desbotamento severo e oclusão.** Película desbotada reduz a saturação abaixo da porta
   `s_min`; oclusão por galho ou poste fragmenta o contorno e derruba a solidez. Ambos
   produzem falsos negativos que nenhum ajuste de limiar recupera sem inundar a máscara.

6. **Sem identificação do significado da placa.** O pipeline entrega forma e posição,
   não a categoria (A-1a, R-19, ...). Essa é, por definição, a tarefa da 2ª Etapa.

### O que a 2ª Etapa acrescenta

| Limitação | Tratamento previsto na N2 |
|---|---|
| Falsos positivos cromáticos | Detector YOLO treinado: aprende contexto e textura, não só cor |
| Placas verdes e desbotadas | Aprendizado supervisionado sobre exemplos reais anotados |
| Ambiguidade círculo/octógono | Classificação por CNN sobre a ROI, e não por descritor geométrico |
| Significado da placa | CNN classificadora treinada no GTSRB (43 categorias) |

O pipeline desta etapa **permanece em produção** na N2 em três papéis: a correção de
iluminação como pré-processamento do detector, o filtro por área mínima como pós-processamento
das caixas propostas, e a contagem clássica como **linha de base comparativa** — é contra os
números da Seção 8 que o ganho do modelo treinado será medido, por mAP, IoU e acurácia.

---

### Próximos passos até a defesa (29/09 e 02/10)

- [ ] Rodar o notebook completo e versionar `outputs/` no repositório Git
- [ ] Colar a Seção "Parâmetros adotados" gerada na Seção 10 dentro do `README.md`
- [ ] Redigir o Relatório Técnico Parcial em PDF a partir das figuras de `outputs/figuras/`
- [ ] Gravar a demonstração em vídeo ou preparar a execução ao vivo
- [ ] Incorporar as imagens de captura autoral (Goiânia-GO) via `PASTA_LOCAL` na Seção 1